# Observable Study: Latent-Observable Correlations

This notebook loads one trained CNN-VAE checkpoint, computes image-based jet observables from the corresponding single-channel pT-normalized jet images, and studies correlations with encoder posterior means `mu(x)`. The previous multi-checkpoint latent effective-dimension scan has been removed from this copy to keep the workflow focused on one model.

In [ ]:
from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "code_training").exists():
    PROJECT_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "code_training").exists():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    raise RuntimeError("Could not locate vae_cnn project root. Run this notebook from vae_cnn/ or one direct child directory of vae_cnn/.")
CODE_DIR = PROJECT_ROOT / "code_training"
sys.path.insert(0, str(CODE_DIR))

from betaVae import BetaVAE
from jet_image_utils import load_events_from_dat, event_to_jet_image, event_to_jet_image_no_pid, pt_normalize, build_pid_channel_map, build_pid_lookup

plt.rcParams["figure.dpi"] = 120
torch.set_grad_enabled(False)
print("notebook dir:", NOTEBOOK_DIR)
print("project root:", PROJECT_ROOT)
PROJECT_ROOT


## Configuration

In [ ]:
DATA_PATH = PROJECT_ROOT.parent / "data" / "u_jet_real_sample.dat"
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "data" / "u_jet_real_sample.dat"

# Select exactly one trained CNN-VAE checkpoint for this observable/correlation study.
CKPT_PATH = PROJECT_ROOT / "results" / "200GeVjet_nopid_annealing" / "mse_all" / "beta_0p000001" / "best_model_betavae.pth"
if not CKPT_PATH.exists():
    candidate_checkpoints = sorted(
        PROJECT_ROOT.glob("results/200GeVjet_nopid_annealing/mse_all/*/best_model_betavae.pth")
    )
    if not candidate_checkpoints:
        raise FileNotFoundError("No fallback checkpoints found under results/200GeVjet_nopid_annealing/mse_all/*/best_model_betavae.pth")
    CKPT_PATH = candidate_checkpoints[-1]

OUTPUT_DIR = NOTEBOOK_DIR / "observable_study_outputs"

BATCH_SIZE = 512
NUM_WORKERS = 0
MAX_EVENTS = None  # set an integer for a faster quick test, e.g. 5000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert DATA_PATH.exists(), f"Data file not found: {DATA_PATH}"
assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("data      :", DATA_PATH)
print("checkpoint:", CKPT_PATH)
print("device    :", DEVICE)
print("output dir:", OUTPUT_DIR)


## Helper Functions

In [ ]:
class JetImageDataset(Dataset):
    def __init__(self, events, pid_lookup, pid_min, n_channels, img_size, checkpoint):
        self.events = events
        self.pid_lookup = pid_lookup
        self.pid_min = pid_min
        self.n_channels = n_channels
        self.img_size = img_size
        self.checkpoint = checkpoint

    def __len__(self):
        return len(self.events)

    def __getitem__(self, idx):
        if checkpoint_uses_no_pid_images(self.checkpoint):
            img = event_to_jet_image_no_pid(
                self.events[idx],
                eta_bins=self.img_size, phi_bins=self.img_size,
                eta_max=0.5, phi_max=0.5,
            )
        else:
            img = event_to_jet_image(
                self.events[idx], self.pid_lookup, self.pid_min, self.n_channels,
                eta_bins=self.img_size, phi_bins=self.img_size,
                eta_max=0.5, phi_max=0.5,
            )
        img = pt_normalize(img).astype(np.float32)
        if checkpoint_uses_standardized_images(self.checkpoint):
            mean_img = np.asarray(self.checkpoint["mean_img"], dtype=np.float32)
            std_img = np.asarray(self.checkpoint["std_img"], dtype=np.float32)
            img = ((img - mean_img) / (std_img + 1e-5)).astype(np.float32)
        return torch.from_numpy(img)


def pid_map_from_checkpoint_or_events(checkpoint, events):
    if "pid_channel_map" in checkpoint:
        return {int(pid): int(ch) for pid, ch in checkpoint["pid_channel_map"].items()}
    return build_pid_channel_map(events)


def checkpoint_hidden_dims(checkpoint):
    return list(checkpoint.get("hidden_dims", [32, 64, 128, 256, 512]))


def checkpoint_output_mode(checkpoint):
    if "output_mode" in checkpoint:
        return str(checkpoint["output_mode"])
    if checkpoint.get("normalization", None) == "event_pt":
        return "event_pt"
    return "tanh"


def checkpoint_channel_mode(checkpoint):
    if "channel_mode" in checkpoint:
        return str(checkpoint["channel_mode"])
    pid_map = checkpoint.get("pid_channel_map", {})
    pid_map = {int(pid): int(ch) for pid, ch in pid_map.items()} if hasattr(pid_map, "items") else {}
    if int(checkpoint["n_channels"]) == 1 and (not pid_map or pid_map == {0: 0}):
        return "eta_phi_total_pt_no_pid"
    return "pid_channels"


def checkpoint_uses_no_pid_images(checkpoint):
    return checkpoint_channel_mode(checkpoint) == "eta_phi_total_pt_no_pid"


def checkpoint_uses_standardized_images(checkpoint):
    return "mean_img" in checkpoint and "std_img" in checkpoint and checkpoint.get("normalization", None) != "event_pt"


def load_model_from_checkpoint(checkpoint, device):
    model = BetaVAE(
        in_channels=int(checkpoint["n_channels"]),
        latent_dim=int(checkpoint["latent_dim"]),
        image_size=int(checkpoint["img_size"]),
        hidden_dims=checkpoint_hidden_dims(checkpoint),
        output_mode=checkpoint_output_mode(checkpoint),
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model


def collect_latent_mu(model, dataset, batch_size, num_workers, device):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    chunks = []
    for batch in loader:
        x = batch.to(device)
        mu, _ = model.encode(x)
        chunks.append(mu.detach().cpu().numpy())
    return np.concatenate(chunks, axis=0)


def collect_latent_mu_logvar(model, dataset, batch_size, num_workers, device):
    """Collect deterministic encoder posterior means and log-variances for a dataset."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    mu_chunks = []
    logvar_chunks = []
    for batch in loader:
        x = batch.to(device)
        mu, logvar = model.encode(x)
        mu_chunks.append(mu.detach().cpu().numpy())
        logvar_chunks.append(logvar.detach().cpu().numpy())
    return np.concatenate(mu_chunks, axis=0), np.concatenate(logvar_chunks, axis=0)


## Load Events

In [ ]:
events = load_events_from_dat(str(DATA_PATH))
if MAX_EVENTS is not None:
    events = events[:MAX_EVENTS]
print("events:", len(events))

## Priority 1 Latent-Observable Extension

This section extends the existing latent effective-dimension notebook with image-based observables and latent-observable correlation analysis. It is appended after the original analysis to preserve backward compatibility.

Implemented here: existing/basic image observables needed for context, S2 two-point spectrum, pair-distance statistics, angular multipoles, Renyi effective multiplicities, threshold occupancy, Minkowski functionals, observable metadata, raw latent-observable correlations, and correlations after binned residualization against `girth`.

Priority 2 shower/texture blocks are implemented in later cells. Priority 3 blocks such as EFPs, Soft Drop, higher-point extensions, and full regression are implemented in later cells with optional dependencies and heavier validation.

### Observable Extension Configuration

In [ ]:
import inspect
import hashlib
import json
import math
import time
import warnings
from collections import defaultdict

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else (lambda x: x)

try:
    from skimage import measure
    from skimage.morphology import binary_dilation, square
    SKIMAGE_AVAILABLE = True
except Exception as exc:
    SKIMAGE_AVAILABLE = False
    measure = None
    binary_dilation = None
    square = None
    warnings.warn(f"scikit-image is not available; Minkowski functionals will be NaN. Reason: {exc}")

OBS_OUTPUT_DIR = OUTPUT_DIR / "observable_extension"
OBS_TABLE_DIR = OBS_OUTPUT_DIR / "tables"
OBS_HEATMAP_DIR = OBS_OUTPUT_DIR / "heatmaps"
OBS_CACHE_DIR = OBS_OUTPUT_DIR / "cache"
for _p in [OBS_OUTPUT_DIR, OBS_TABLE_DIR, OBS_HEATMAP_DIR, OBS_CACHE_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

# Use the single configured checkpoint for latent-observable correlations.
OBS_CHECKPOINT_PATH = CKPT_PATH

# Runtime controls.
max_jets = None
expensive_max_jets = 10000
random_seed = 42
SIGMA_FEATURE_INDEX = 18
observable_cache_version = "priority1_observables_v1"

# Physics/image geometry. Image coordinate half-ranges are read from the image builder defaults below.
JET_RADIUS = 0.4
_no_pid_signature = inspect.signature(event_to_jet_image_no_pid)
IMAGE_ETA_MAX = float(_no_pid_signature.parameters["eta_max"].default)
IMAGE_PHI_MAX = float(_no_pid_signature.parameters["phi_max"].default)

# Active-pixel and expensive observable controls.
pixel_active_threshold = 1e-12
max_pair_pixels = 512
max_e3_pixels = 50
s2_edges = np.linspace(0.0, 2.0, 21)
s2_clip_overflow_to_last_bin = True
beta_values_e2 = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0]
beta_values_e3 = [0.5, 1.0, 2.0]
ecf_denominator_eps = 1e-14
multipole_n_values = [2, 3, 4, 5, 6]
multipole_p_values = [1, 2, 3]
renyi_q_values = [0, 0.5, 1, 2, 3, 4]
occupancy_thresholds = [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
dilation_steps = [0, 1, 2, 3, 4]
minkowski_connectivity = 2  # 2 means 8-connectivity for 2D skimage labels/euler_number.
residualize_n_bins = 25
rho_eps = 1e-12

observable_config = {
    "checkpoint": str(OBS_CHECKPOINT_PATH),
    "data_path": str(DATA_PATH),
    "max_jets": max_jets,
    "expensive_max_jets": expensive_max_jets,
    "random_seed": random_seed,
    "jet_radius": JET_RADIUS,
    "image_eta_max": IMAGE_ETA_MAX,
    "image_phi_max": IMAGE_PHI_MAX,
    "pixel_active_threshold": pixel_active_threshold,
    "max_pair_pixels": max_pair_pixels,
    "max_e3_pixels": max_e3_pixels,
    "s2_edges": s2_edges.tolist(),
    "s2_clip_overflow_to_last_bin": s2_clip_overflow_to_last_bin,
    "beta_values_e2": beta_values_e2,
    "beta_values_e3": beta_values_e3,
    "multipole_n_values": multipole_n_values,
    "multipole_p_values": multipole_p_values,
    "renyi_q_values": renyi_q_values,
    "occupancy_thresholds": occupancy_thresholds,
    "dilation_steps": dilation_steps,
    "minkowski_connectivity": minkowski_connectivity,
    "residualize_n_bins": residualize_n_bins,
    "version": observable_cache_version,
}
observable_config_hash = hashlib.sha256(json.dumps(observable_config, sort_keys=True).encode("utf-8")).hexdigest()[:16]
print("observable checkpoint:", OBS_CHECKPOINT_PATH.relative_to(PROJECT_ROOT))
print("image half-ranges from event_to_jet_image_no_pid defaults:", IMAGE_ETA_MAX, IMAGE_PHI_MAX)
print("AK jet radius used for normalized coordinates R:", JET_RADIUS)
print("observable config hash:", observable_config_hash)
print("sigma feature included in latent correlations:", SIGMA_FEATURE_INDEX)
print("skimage available:", SKIMAGE_AVAILABLE)


### Coordinate Grid and Metadata Utilities

In [ ]:
def add_metadata(rows, observable_name, family, subfamily, description, *, irc_safe=True, rotation_invariant=True, pixelization_sensitive=False, expensive=False, parameters=None):
    """Append one observable metadata row.

    Metadata marks whether a quantity is physics-like or diagnostic, whether it is rotation invariant,
    and which configuration parameters define it.
    """
    rows.append({
        "observable_name": observable_name,
        "family": family,
        "subfamily": subfamily,
        "description": description,
        "irc_safe": bool(irc_safe),
        "rotation_invariant": bool(rotation_invariant),
        "pixelization_sensitive": bool(pixelization_sensitive),
        "expensive": bool(expensive),
        "parameters": json.dumps(parameters or {}, sort_keys=True),
    })

def beta_label(beta):
    return str(beta).replace(".", "p")

def threshold_label(value):
    return f"{value:.0e}".replace("+", "").replace("-", "m")

def build_pixel_coordinate_grid(image_size, eta_max, phi_max, jet_radius):
    """Build pixel-center coordinate grids for images shaped (1, phi, eta).

    Returns physical Delta eta/phi grids and normalized x=Delta eta/R, y=Delta phi/R grids.
    The axis convention follows jet_image_utils.event_to_jet_image_no_pid: image[0, phi_index, eta_index].
    """
    eta_centers = -eta_max + (np.arange(image_size) + 0.5) * (2.0 * eta_max / image_size)
    phi_centers = -phi_max + (np.arange(image_size) + 0.5) * (2.0 * phi_max / image_size)
    eta_grid, phi_grid = np.meshgrid(eta_centers, phi_centers, indexing="xy")
    x_grid = eta_grid / jet_radius
    y_grid = phi_grid / jet_radius
    return eta_grid, phi_grid, x_grid, y_grid, float(eta_centers[1] - eta_centers[0]), float(phi_centers[1] - phi_centers[0])

def prepare_probability_image(image):
    """Return a non-negative, float64, event-normalized 2D image and diagnostics.

    Input may be (1, phi, eta) or (phi, eta). Invalid non-positive sums are flagged rather than silently zeroed.
    """
    arr = np.asarray(image, dtype=np.float64)
    if arr.ndim == 3:
        if arr.shape[0] != 1:
            raise ValueError(f"Expected single-channel image, got shape {arr.shape}")
        arr = arr[0]
    arr = np.clip(arr, 0.0, None)
    total = float(arr.sum())
    if not np.isfinite(total) or total <= 0.0:
        return arr, {"valid": False, "sum_before_norm": total, "norm_deviation": np.nan}
    z = arr / total
    return z, {"valid": True, "sum_before_norm": total, "norm_deviation": abs(float(z.sum()) - 1.0)}

def make_raw_event_image_for_observables(event, checkpoint):
    """Create the pT-normalized image used for observables, before any model-input standardization."""
    img_size_local = int(checkpoint["img_size"])
    n_channels_local = int(checkpoint["n_channels"])
    if checkpoint_uses_no_pid_images(checkpoint):
        img = event_to_jet_image_no_pid(event, eta_bins=img_size_local, phi_bins=img_size_local, eta_max=IMAGE_ETA_MAX, phi_max=IMAGE_PHI_MAX)
    else:
        pid_channel_map = pid_map_from_checkpoint_or_events(checkpoint, events)
        pid_lookup_local, pid_min_local = build_pid_lookup(pid_channel_map)
        img = event_to_jet_image(event, pid_lookup_local, pid_min_local, n_channels_local, eta_bins=img_size_local, phi_bins=img_size_local, eta_max=IMAGE_ETA_MAX, phi_max=IMAGE_PHI_MAX)
    return pt_normalize(img).astype(np.float32)


### Load Selected Images and Latent Means

In [ ]:
obs_start_time = time.perf_counter()
obs_checkpoint = torch.load(OBS_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
obs_model = load_model_from_checkpoint(obs_checkpoint, DEVICE)
obs_img_size = int(obs_checkpoint["img_size"])
obs_n_channels = int(obs_checkpoint["n_channels"])
obs_latent_dim = int(obs_checkpoint["latent_dim"])
if obs_n_channels != 1:
    warnings.warn(f"Expected single-channel images for this extension, got n_channels={obs_n_channels}.")
if SIGMA_FEATURE_INDEX < 0 or SIGMA_FEATURE_INDEX >= obs_latent_dim:
    raise ValueError(f"SIGMA_FEATURE_INDEX={SIGMA_FEATURE_INDEX} is outside latent_dim={obs_latent_dim}.")
ETA_GRID, PHI_GRID, X_GRID, Y_GRID, ETA_PIXEL_SIZE, PHI_PIXEL_SIZE = build_pixel_coordinate_grid(obs_img_size, IMAGE_ETA_MAX, IMAGE_PHI_MAX, JET_RADIUS)
print("image shape convention: image[0, phi_index, eta_index]")
print("physical eta centers range:", float(ETA_GRID.min()), float(ETA_GRID.max()), "pixel size:", ETA_PIXEL_SIZE)
print("physical phi centers range:", float(PHI_GRID.min()), float(PHI_GRID.max()), "pixel size:", PHI_PIXEL_SIZE)
print("normalized x range:", float(X_GRID.min()), float(X_GRID.max()))
print("normalized y range:", float(Y_GRID.min()), float(Y_GRID.max()))

rng = np.random.default_rng(random_seed)
all_indices = np.arange(len(events), dtype=np.int64)
if max_jets is None or max_jets >= len(events):
    observable_event_indices = all_indices
else:
    observable_event_indices = np.sort(rng.choice(all_indices, size=int(max_jets), replace=False))
print("jets selected for observable extension:", len(observable_event_indices))

latent_cache_path = OBS_CACHE_DIR / f"latent_encoder_{observable_config_hash}.npz"
image_cache_path = OBS_CACHE_DIR / f"images_{observable_config_hash}.npz"

if latent_cache_path.exists() and image_cache_path.exists():
    cached_latent = np.load(latent_cache_path)
    latent_mu_obs = cached_latent["mu"]
    latent_logvar_obs = cached_latent["logvar"]
    normalized_images = np.load(image_cache_path)["images"]
    print("loaded latent means/logvars and images from cache")
else:
    pid_channel_map = pid_map_from_checkpoint_or_events(obs_checkpoint, events)
    pid_lookup, pid_min = build_pid_lookup(pid_channel_map)
    selected_events = [events[int(i)] for i in observable_event_indices]
    obs_dataset = JetImageDataset(selected_events, pid_lookup, pid_min, obs_n_channels, obs_img_size, obs_checkpoint)
    latent_mu_obs, latent_logvar_obs = collect_latent_mu_logvar(obs_model, obs_dataset, BATCH_SIZE, NUM_WORKERS, DEVICE)
    normalized_images = np.stack([
        make_raw_event_image_for_observables(events[int(i)], obs_checkpoint)[0]
        for i in tqdm(observable_event_indices, desc="Build normalized images")
    ]).astype(np.float32)
    np.savez_compressed(latent_cache_path, mu=latent_mu_obs, logvar=latent_logvar_obs, event_indices=observable_event_indices)
    np.savez_compressed(image_cache_path, images=normalized_images, event_indices=observable_event_indices)
    print("saved latent means/logvars and images to cache")

latent_sigma_obs = np.exp(0.5 * latent_logvar_obs.astype(np.float64))
mu_feature_names = [f"mu_{i}" for i in range(latent_mu_obs.shape[1])]
sigma_feature_name = f"sigma_{SIGMA_FEATURE_INDEX}"
latent_correlation_feature_names = mu_feature_names + [sigma_feature_name]
latent_mu_df = pd.DataFrame(latent_mu_obs, columns=mu_feature_names)
latent_mu_df[sigma_feature_name] = latent_sigma_obs[:, SIGMA_FEATURE_INDEX]
latent_mu_df.insert(0, "event_index", observable_event_indices)
print("latent_mu_obs shape:", latent_mu_obs.shape)
print("latent_logvar_obs shape:", latent_logvar_obs.shape)
print("included sigma column:", sigma_feature_name)
print("latent correlation rows:", len(latent_correlation_feature_names))
print("normalized_images shape:", normalized_images.shape)


### Priority 1 Observable Functions

In [ ]:
def select_active_pixels(z_img, x_grid, y_grid, *, threshold=pixel_active_threshold, max_pixels=None):
    """Select active pixels and optionally keep only the largest-z pixels without renormalizing.

    Returns z, x, y arrays. Truncation is an explicit approximation for expensive pair/triplet observables.
    """
    z = z_img.ravel()
    x = x_grid.ravel()
    y = y_grid.ravel()
    mask = z > threshold
    z = z[mask].astype(np.float64)
    x = x[mask].astype(np.float64)
    y = y[mask].astype(np.float64)
    if max_pixels is not None and len(z) > max_pixels:
        order = np.argsort(z)[::-1][:int(max_pixels)]
        z = z[order]
        x = x[order]
        y = y[order]
    return z, x, y

def pairwise_distances_and_weights(z, x, y):
    """Return upper-triangle pair distances d_ab and weights z_a z_b."""
    n = len(z)
    if n < 2:
        return np.array([], dtype=np.float64), np.array([], dtype=np.float64)
    i, j = np.triu_indices(n, k=1)
    d = np.sqrt((x[i] - x[j]) ** 2 + (y[i] - y[j]) ** 2)
    w = z[i] * z[j]
    return d, w

def weighted_quantile(values, weights, quantiles):
    """Weighted quantiles for one-dimensional finite arrays."""
    values = np.asarray(values, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if mask.sum() == 0:
        return np.full(len(quantiles), np.nan)
    values = values[mask]
    weights = weights[mask]
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cdf = np.cumsum(weights) / np.sum(weights)
    return np.interp(quantiles, cdf, values)

def compute_basic_observables(z_img, eta_grid, phi_grid, x_grid, y_grid):
    """Compute existing/basic image observables from a normalized single-channel image.

    Includes log_rho, LHA, girth, radial quantiles, e2_beta_1, approximate D2_beta_1,
    eccentricity, principal-axis angle, effective pixel multiplicity, and Shannon entropy.
    """
    z_all = z_img.ravel().astype(np.float64)
    eta_all = eta_grid.ravel().astype(np.float64)
    phi_all = phi_grid.ravel().astype(np.float64)
    x_all = x_grid.ravel().astype(np.float64)
    y_all = y_grid.ravel().astype(np.float64)
    pos = z_all > 0.0
    z = z_all[pos]
    eta = eta_all[pos]
    phi = phi_all[pos]
    x = x_all[pos]
    y = y_all[pos]
    if len(z) == 0:
        return {}
    xbar = float(np.sum(z * x))
    ybar = float(np.sum(z * y))
    dx = x - xbar
    dy = y - ybar
    r = np.sqrt(dx ** 2 + dy ** 2)
    e_tilde = float(np.sum(z * np.cosh(eta)))
    px_tilde = float(np.sum(z * np.cos(phi)))
    py_tilde = float(np.sum(z * np.sin(phi)))
    pz_tilde = float(np.sum(z * np.sinh(eta)))
    m2_tilde = max(e_tilde ** 2 - px_tilde ** 2 - py_tilde ** 2 - pz_tilde ** 2, 0.0)
    pt2_tilde = max(px_tilde ** 2 + py_tilde ** 2, np.finfo(np.float64).tiny)
    rho = m2_tilde / (pt2_tilde * JET_RADIUS ** 2)
    order = np.argsort(r, kind="mergesort")
    cum = np.cumsum(z[order])
    r50 = float(r[order][np.searchsorted(cum, 0.5, side="left")])
    r80 = float(r[order][np.searchsorted(cum, 0.8, side="left")])
    z_pair, x_pair, y_pair = select_active_pixels(z_img, x_grid, y_grid, max_pixels=max_e3_pixels)
    d_pair, w_pair = pairwise_distances_and_weights(z_pair, x_pair, y_pair)
    e2_beta_1 = float(np.sum(w_pair * d_pair)) if len(d_pair) else 0.0
    e3_beta_1 = compute_e3_beta(z_pair, x_pair, y_pair, beta=1.0) if len(z_pair) >= 3 else 0.0
    D2_beta_1 = np.nan if e2_beta_1 <= ecf_denominator_eps else float(e3_beta_1 / (e2_beta_1 ** 3))
    cxx = float(np.sum(z * dx ** 2))
    cyy = float(np.sum(z * dy ** 2))
    cxy = float(np.sum(z * dx * dy))
    trace = cxx + cyy
    discr = math.sqrt(max((cxx - cyy) ** 2 + 4.0 * cxy ** 2, 0.0))
    lambda1 = 0.5 * (trace + discr)
    lambda2 = 0.5 * (trace - discr)
    eccentricity = (lambda1 - lambda2) / max(lambda1 + lambda2, np.finfo(np.float64).tiny)
    alpha = 0.5 * math.atan2(2.0 * cxy, cxx - cyy)
    if alpha >= math.pi / 2:
        alpha -= math.pi
    if alpha < -math.pi / 2:
        alpha += math.pi
    n_active = int(np.count_nonzero(z_img > pixel_active_threshold))
    sum_z2 = float(np.sum(z ** 2))
    entropy = float(-np.sum(z * np.log(z)))
    return {
        "rho": float(rho),
        "log_rho": float(np.log(rho + rho_eps)),
        "LHA": float(np.sum(z * np.sqrt(np.maximum(r, 0.0)))),
        "girth": float(np.sum(z * r)),
        "R50_over_R": r50,
        "R80_over_R": r80,
        "e2_beta_1": e2_beta_1,
        "D2_beta_1": D2_beta_1,
        "eccentricity": float(np.clip(eccentricity, 0.0, 1.0)),
        "principal_axis_angle": float(alpha),
        "cos_2alpha_principal": float(np.cos(2.0 * alpha)),
        "sin_2alpha_principal": float(np.sin(2.0 * alpha)),
        "effective_pixel_multiplicity": float(1.0 / max(sum_z2, np.finfo(np.float64).tiny)),
        "image_entropy": entropy,
        "n_active_pixels": n_active,
    }

def compute_s2_observables(z_img, x_grid, y_grid):
    """Compute binned S2(D), pair-distance moments, and weighted pair-distance quantiles.

    Distances use normalized coordinates d_ab=sqrt((x_a-x_b)^2+(y_a-y_b)^2). If configured,
    distances above the final S2 edge are counted in the final bin and the overflow weight is recorded.
    """
    z, x, y = select_active_pixels(z_img, x_grid, y_grid, max_pixels=max_pair_pixels)
    d, w = pairwise_distances_and_weights(z, x, y)
    out = {"s2_selected_energy_fraction": float(np.sum(z)), "s2_n_selected_pixels": int(len(z))}
    if len(d) == 0 or np.sum(w) <= 0:
        for k in range(len(s2_edges) - 1):
            out[f"s2_bin_{k:02d}"] = 0.0
        for name in ["pair_distance_mean", "pair_distance_std", "pair_distance_skewness", "pair_distance_kurtosis", "pair_D10", "pair_D25", "pair_D50", "pair_D75", "pair_D90"]:
            out[name] = np.nan
        out["s2_total_pair_weight"] = 0.0
        out["s2_hist_sum"] = 0.0
        out["s2_overflow_ge_last_edge"] = 0.0
        return out
    overflow = float(np.sum(w[d >= s2_edges[-1]]))
    d_for_hist = d.copy()
    if s2_clip_overflow_to_last_bin:
        d_for_hist = np.minimum(d_for_hist, np.nextafter(s2_edges[-1], s2_edges[0]))
    hist, _ = np.histogram(d_for_hist, bins=s2_edges, weights=w)
    for k, value in enumerate(hist):
        out[f"s2_bin_{k:02d}"] = float(value)
    wsum = float(np.sum(w))
    p = w / wsum
    mean = float(np.sum(p * d))
    var = float(np.sum(p * (d - mean) ** 2))
    std = math.sqrt(max(var, 0.0))
    if std > 0:
        skew = float(np.sum(p * ((d - mean) / std) ** 3))
        kurt = float(np.sum(p * ((d - mean) / std) ** 4))
    else:
        skew = np.nan
        kurt = np.nan
    q10, q25, q50, q75, q90 = weighted_quantile(d, w, [0.10, 0.25, 0.50, 0.75, 0.90])
    out.update({
        "pair_distance_mean": mean,
        "pair_distance_std": std,
        "pair_distance_skewness": skew,
        "pair_distance_kurtosis": kurt,
        "pair_D10": float(q10),
        "pair_D25": float(q25),
        "pair_D50": float(q50),
        "pair_D75": float(q75),
        "pair_D90": float(q90),
        "s2_total_pair_weight": wsum,
        "s2_hist_sum": float(np.sum(hist)),
        "s2_overflow_ge_last_edge": overflow,
    })
    return out

def compute_e2_observables(z_img, x_grid, y_grid):
    """Compute two-point energy correlators e2_beta for configured beta values."""
    z, x, y = select_active_pixels(z_img, x_grid, y_grid, max_pixels=max_pair_pixels)
    d, w = pairwise_distances_and_weights(z, x, y)
    out = {}
    for beta in beta_values_e2:
        out[f"e2_beta_{beta_label(beta)}"] = float(np.sum(w * (d ** beta))) if len(d) else 0.0
    return out

def compute_e3_beta(z, x, y, beta):
    """Approximate e3_beta using selected active pixels only; intended for capped max_e3_pixels."""
    n = len(z)
    total = 0.0
    for a in range(n - 2):
        for b in range(a + 1, n - 1):
            dab = math.hypot(float(x[a] - x[b]), float(y[a] - y[b]))
            for c in range(b + 1, n):
                dac = math.hypot(float(x[a] - x[c]), float(y[a] - y[c]))
                dbc = math.hypot(float(x[b] - x[c]), float(y[b] - y[c]))
                total += float(z[a] * z[b] * z[c]) * ((dab * dac * dbc) ** beta)
    return float(total)

def compute_e3_c2_d2_observables(z_img, x_grid, y_grid):
    """Compute approximate e3, C2, and D2 for configured beta values with max_e3_pixels truncation."""
    z, x, y = select_active_pixels(z_img, x_grid, y_grid, max_pixels=max_e3_pixels)
    e2 = compute_e2_observables(z_img, x_grid, y_grid)
    out = {"e3_selected_energy_fraction": float(np.sum(z)), "e3_n_selected_pixels": int(len(z))}
    for beta in beta_values_e3:
        label = beta_label(beta)
        e3 = compute_e3_beta(z, x, y, beta) if len(z) >= 3 else 0.0
        e2_value = e2.get(f"e2_beta_{label}", np.nan)
        out[f"e3_beta_{label}"] = e3
        out[f"C2_beta_{label}"] = np.nan if not np.isfinite(e2_value) or e2_value <= ecf_denominator_eps else float(e3 / (e2_value ** 2))
        out[f"D2_beta_{label}"] = np.nan if not np.isfinite(e2_value) or e2_value <= ecf_denominator_eps else float(e3 / (e2_value ** 3))
    return out

def compute_multipole_observables(z_img, x_grid, y_grid):
    """Compute angular multipole magnitudes and diagnostic phases from pT-weighted pixels."""
    z, x, y = select_active_pixels(z_img, x_grid, y_grid, max_pixels=None)
    r = np.sqrt(x ** 2 + y ** 2)
    phi = np.arctan2(y, x)
    out = {}
    for pval in multipole_p_values:
        radial_weight = z * (r ** pval)
        denom = float(np.sum(radial_weight))
        for nval in multipole_n_values:
            key = f"multipole_n{nval}_p{pval}"
            if denom <= 0:
                out[key] = np.nan
                out[f"{key}_cos_phase"] = np.nan
                out[f"{key}_sin_phase"] = np.nan
            else:
                complex_moment = np.sum(radial_weight * np.exp(1j * nval * phi))
                out[key] = float(np.abs(complex_moment) / denom)
                phase = np.angle(complex_moment)
                out[f"{key}_cos_phase"] = float(np.cos(phase))
                out[f"{key}_sin_phase"] = float(np.sin(phase))
    return out

def compute_renyi_observables(z_img):
    """Compute Renyi effective multiplicities D_q, including D0, D1, D2, and Dinf."""
    z = z_img.ravel().astype(np.float64)
    z = z[z > 0.0]
    out = {}
    entropy = float(-np.sum(z * np.log(z))) if len(z) else np.nan
    for q in renyi_q_values:
        label = beta_label(q)
        if q == 0:
            value = float(len(z))
        elif q == 1:
            value = float(np.exp(entropy))
        else:
            value = float(np.sum(z ** q) ** (1.0 / (1.0 - q)))
        out[f"renyi_D{label}"] = value
    out["renyi_Dinf"] = float(1.0 / np.max(z)) if len(z) else np.nan
    return out

def compute_occupancy_observables(z_img):
    """Count pixels above fixed intensity thresholds."""
    z = z_img.ravel().astype(np.float64)
    return {f"occupancy_t{threshold_label(t)}": int(np.count_nonzero(z > t)) for t in occupancy_thresholds}

def compute_minkowski_observables(z_img):
    """Compute thresholded binary-image Minkowski and connectivity observables.

    Uses skimage with fixed 8-connectivity when available. Results are pixelization-sensitive by construction.
    """
    out = {}
    if not SKIMAGE_AVAILABLE:
        for t in occupancy_thresholds:
            for dstep in dilation_steps:
                prefix = f"minkowski_t{threshold_label(t)}_d{dstep}"
                for suffix in ["area", "perimeter", "euler", "n_components", "n_holes", "largest_area_frac", "largest_energy_frac", "second_area_frac", "second_energy_frac"]:
                    out[f"{prefix}_{suffix}"] = np.nan
        return out
    for t in occupancy_thresholds:
        base_mask = z_img > t
        for dstep in dilation_steps:
            mask = base_mask.copy()
            for _ in range(int(dstep)):
                mask = binary_dilation(mask, footprint=square(3))
            prefix = f"minkowski_t{threshold_label(t)}_d{dstep}"
            labels = measure.label(mask, connectivity=minkowski_connectivity)
            n_components = int(labels.max())
            euler = float(measure.euler_number(mask, connectivity=minkowski_connectivity))
            n_holes = float(n_components - euler)
            area = float(mask.sum())
            perimeter = float(measure.perimeter(mask, neighborhood=8)) if area > 0 else 0.0
            component_area_fracs = []
            component_energy_fracs = []
            for label_id in range(1, n_components + 1):
                comp = labels == label_id
                component_area_fracs.append(float(comp.sum() / max(area, 1.0)))
                component_energy_fracs.append(float(z_img[comp].sum()))
            order = np.argsort(component_area_fracs)[::-1] if component_area_fracs else []
            largest_area = component_area_fracs[order[0]] if len(component_area_fracs) else np.nan
            largest_energy = component_energy_fracs[order[0]] if len(component_energy_fracs) else np.nan
            second_area = component_area_fracs[order[1]] if len(component_area_fracs) > 1 else np.nan
            second_energy = component_energy_fracs[order[1]] if len(component_energy_fracs) > 1 else np.nan
            out.update({
                f"{prefix}_area": area,
                f"{prefix}_perimeter": perimeter,
                f"{prefix}_euler": euler,
                f"{prefix}_n_components": n_components,
                f"{prefix}_n_holes": n_holes,
                f"{prefix}_largest_area_frac": largest_area,
                f"{prefix}_largest_energy_frac": largest_energy,
                f"{prefix}_second_area_frac": second_area,
                f"{prefix}_second_energy_frac": second_energy,
            })
    return out

def compute_diagnostic_observables(z_img, x_grid, y_grid):
    """Compute artifact and geometry diagnostics, not primary physics observables."""
    z = z_img.ravel().astype(np.float64)
    x = x_grid.ravel().astype(np.float64)
    y = y_grid.ravel().astype(np.float64)
    pos = z > 0.0
    xbar = float(np.sum(z * x))
    ybar = float(np.sum(z * y))
    r = np.sqrt(x ** 2 + y ** 2)
    boundary = np.zeros_like(z_img, dtype=bool)
    boundary[0, :] = True
    boundary[-1, :] = True
    boundary[:, 0] = True
    boundary[:, -1] = True
    return {
        "centroid_x": xbar,
        "centroid_y": ybar,
        "centroid_r": float(math.hypot(xbar, ybar)),
        "energy_outside_r1": float(np.sum(z[r > 1.0])),
        "boundary_energy_fraction": float(np.sum(z_img[boundary])),
        "grid_multipole_n4_p2": compute_multipole_observables(z_img, x_grid, y_grid).get("multipole_n4_p2", np.nan),
        "n_exact_zero_pixels": int(np.count_nonzero(z == 0.0)),
        "min_nonzero_pixel_intensity": float(np.min(z[pos])) if np.any(pos) else np.nan,
        "max_pixel_intensity": float(np.max(z)) if len(z) else np.nan,
    }


### Synthetic-Image Sanity Tests

In [ ]:
def synthetic_image(points):
    """Build a normalized synthetic image from (phi_index, eta_index, weight) tuples."""
    img = np.zeros((obs_img_size, obs_img_size), dtype=np.float64)
    for phi_idx, eta_idx, weight in points:
        img[int(phi_idx), int(eta_idx)] += float(weight)
    if img.sum() > 0:
        img /= img.sum()
    return img

center_phi = obs_img_size // 2
center_eta = obs_img_size // 2
single_center = synthetic_image([(center_phi, center_eta, 1.0)])
two_pixel = synthetic_image([(center_phi, max(center_eta - 4, 0), 1.0), (center_phi, min(center_eta + 4, obs_img_size - 1), 1.0)])
line_eta = synthetic_image([(center_phi, max(center_eta - 6, 0), 1.0), (center_phi, center_eta, 1.0), (center_phi, min(center_eta + 6, obs_img_size - 1), 1.0)])
line_phi = synthetic_image([(max(center_phi - 6, 0), center_eta, 1.0), (center_phi, center_eta, 1.0), (min(center_phi + 6, obs_img_size - 1), center_eta, 1.0)])
block = synthetic_image([(center_phi + dp, center_eta + de, 1.0) for dp in [-1, 0, 1] for de in [-1, 0, 1]])

test_rows = []
for name, img in [("single_center", single_center), ("two_pixel", two_pixel), ("line_eta", line_eta), ("line_phi", line_phi), ("block", block)]:
    basic = compute_basic_observables(img, ETA_GRID, PHI_GRID, X_GRID, Y_GRID)
    s2 = compute_s2_observables(img, X_GRID, Y_GRID)
    renyi = compute_renyi_observables(img)
    multipoles = compute_multipole_observables(img, X_GRID, Y_GRID)
    minkowski = compute_minkowski_observables(img)
    test_rows.append({"synthetic_image": name, **basic, **s2, **renyi, **multipoles, **minkowski})
synthetic_test_df = pd.DataFrame(test_rows)

assert abs(single_center.sum() - 1.0) < 1e-12
assert abs(two_pixel.sum() - 1.0) < 1e-12
assert synthetic_test_df.loc[synthetic_test_df.synthetic_image == "single_center", "effective_pixel_multiplicity"].iloc[0] == 1.0
assert abs(synthetic_test_df.loc[synthetic_test_df.synthetic_image == "single_center", "image_entropy"].iloc[0]) < 1e-12
assert abs(synthetic_test_df.loc[synthetic_test_df.synthetic_image == "two_pixel", "effective_pixel_multiplicity"].iloc[0] - 2.0) < 1e-12
assert abs(synthetic_test_df.loc[synthetic_test_df.synthetic_image == "two_pixel", "image_entropy"].iloc[0] - math.log(2.0)) < 1e-12
assert np.nanmax(synthetic_test_df[[c for c in synthetic_test_df.columns if c.startswith("multipole_n") and not c.endswith(("cos_phase", "sin_phase"))]].to_numpy(dtype=float)) <= 1.0 + 1e-12
if SKIMAGE_AVAILABLE:
    component_col = f"minkowski_t{threshold_label(occupancy_thresholds[0])}_d0_n_components"
    assert synthetic_test_df.loc[synthetic_test_df.synthetic_image == "single_center", component_col].iloc[0] == 1
s2_cols = [f"s2_bin_{k:02d}" for k in range(len(s2_edges) - 1)]
s2_diff = np.abs(synthetic_test_df[s2_cols].sum(axis=1) - synthetic_test_df["s2_total_pair_weight"])
assert np.nanmax(s2_diff) < 1e-12

display(synthetic_test_df[["synthetic_image", "girth", "effective_pixel_multiplicity", "image_entropy", "s2_total_pair_weight", "pair_D50"]])
print("Synthetic-image sanity tests passed.")


### Observable Computation and Metadata

In [ ]:
metadata_rows = []
for name in ["rho", "log_rho", "LHA", "girth", "R50_over_R", "R80_over_R", "e2_beta_1", "D2_beta_1", "eccentricity", "effective_pixel_multiplicity", "image_entropy"]:
    add_metadata(metadata_rows, name, "existing_radial", "basic", f"Basic image observable {name}")
for name in ["principal_axis_angle", "cos_2alpha_principal", "sin_2alpha_principal"]:
    add_metadata(metadata_rows, name, "diagnostic", "principal_axis", f"Principal-axis diagnostic {name}", rotation_invariant=False)
for k in range(len(s2_edges) - 1):
    add_metadata(metadata_rows, f"s2_bin_{k:02d}", "energy_flow", "S2", "Binned two-point energy-flow spectrum S2(D)", parameters={"bin": k, "edges": [float(s2_edges[k]), float(s2_edges[k + 1])]})
for name in ["pair_distance_mean", "pair_distance_std", "pair_distance_skewness", "pair_distance_kurtosis", "pair_D10", "pair_D25", "pair_D50", "pair_D75", "pair_D90"]:
    add_metadata(metadata_rows, name, "energy_flow", "pair_distance", f"Weighted pair-distance statistic {name}")
for beta in beta_values_e2:
    add_metadata(metadata_rows, f"e2_beta_{beta_label(beta)}", "energy_flow", "e2", "Two-point energy correlator", parameters={"beta": beta})
for beta in beta_values_e3:
    label = beta_label(beta)
    for prefix in ["e3_beta", "C2_beta", "D2_beta"]:
        add_metadata(metadata_rows, f"{prefix}_{label}", "energy_flow", "e3_c2_d2", f"Approximate {prefix} with active-pixel truncation", expensive=True, parameters={"beta": beta, "max_e3_pixels": max_e3_pixels})
for pval in multipole_p_values:
    for nval in multipole_n_values:
        base = f"multipole_n{nval}_p{pval}"
        add_metadata(metadata_rows, base, "energy_flow", "multipole", "Rotationally invariant angular multipole magnitude", parameters={"n": nval, "p": pval})
        add_metadata(metadata_rows, f"{base}_cos_phase", "diagnostic", "multipole_phase", "Diagnostic multipole phase cosine", rotation_invariant=False, parameters={"n": nval, "p": pval})
        add_metadata(metadata_rows, f"{base}_sin_phase", "diagnostic", "multipole_phase", "Diagnostic multipole phase sine", rotation_invariant=False, parameters={"n": nval, "p": pval})
for q in renyi_q_values:
    add_metadata(metadata_rows, f"renyi_D{beta_label(q)}", "morphology", "renyi", "Renyi effective multiplicity", pixelization_sensitive=True, parameters={"q": q})
add_metadata(metadata_rows, "renyi_Dinf", "morphology", "renyi", "Inverse leading-pixel intensity", pixelization_sensitive=True)
for t in occupancy_thresholds:
    add_metadata(metadata_rows, f"occupancy_t{threshold_label(t)}", "morphology", "occupancy", "Thresholded occupancy", pixelization_sensitive=True, parameters={"threshold": t})
for t in occupancy_thresholds:
    for dstep in dilation_steps:
        prefix = f"minkowski_t{threshold_label(t)}_d{dstep}"
        for suffix in ["area", "perimeter", "euler", "n_components", "n_holes", "largest_area_frac", "largest_energy_frac", "second_area_frac", "second_energy_frac"]:
            add_metadata(metadata_rows, f"{prefix}_{suffix}", "morphology", "minkowski", "Thresholded Minkowski/connectivity observable", pixelization_sensitive=True, expensive=True, parameters={"threshold": t, "dilation": dstep, "connectivity": minkowski_connectivity})
for name in ["centroid_x", "centroid_y", "centroid_r", "energy_outside_r1", "boundary_energy_fraction", "grid_multipole_n4_p2", "n_exact_zero_pixels", "min_nonzero_pixel_intensity", "max_pixel_intensity", "s2_selected_energy_fraction", "s2_n_selected_pixels", "e3_selected_energy_fraction", "e3_n_selected_pixels", "s2_overflow_ge_last_edge"]:
    add_metadata(metadata_rows, name, "diagnostic", "artifact", f"Artifact/implementation diagnostic {name}", irc_safe=False, pixelization_sensitive=True)
observable_metadata_df = pd.DataFrame(metadata_rows).drop_duplicates("observable_name")
observable_metadata_df.to_csv(OBS_TABLE_DIR / "observable_metadata.csv", index=False)

obs_cache_path = OBS_CACHE_DIR / f"priority1_observables_{observable_config_hash}.parquet"
obs_cache_csv_path = OBS_CACHE_DIR / f"priority1_observables_{observable_config_hash}.csv"
if obs_cache_path.exists():
    observable_df = pd.read_parquet(obs_cache_path)
    print("loaded observables from parquet cache:", obs_cache_path)
elif obs_cache_csv_path.exists():
    observable_df = pd.read_csv(obs_cache_csv_path)
    print("loaded observables from CSV cache:", obs_cache_csv_path)
else:
    rows = []
    pixel_quantile_sample = normalized_images.reshape(-1)
    positive_pixels = pixel_quantile_sample[pixel_quantile_sample > 0]
    print("positive-pixel intensity quantiles:", np.quantile(positive_pixels, [0.01, 0.05, 0.1, 0.5, 0.9, 0.99]).tolist() if len(positive_pixels) else [])
    if len(observable_event_indices) > expensive_max_jets:
        expensive_indices = set(rng.choice(np.arange(len(observable_event_indices)), size=int(expensive_max_jets), replace=False).tolist())
        print(f"expensive morphology/e3 modules run on {expensive_max_jets} reproducible jets; other rows receive NaN for those columns")
    else:
        expensive_indices = set(range(len(observable_event_indices)))
    for local_idx, event_index in tqdm(list(enumerate(observable_event_indices)), desc="Compute Priority 1 observables"):
        z_img, prep = prepare_probability_image(normalized_images[local_idx])
        row = {"event_index": int(event_index), "image_norm_deviation": prep["norm_deviation"], "image_valid": bool(prep["valid"])}
        if not prep["valid"]:
            rows.append(row)
            continue
        row.update(compute_basic_observables(z_img, ETA_GRID, PHI_GRID, X_GRID, Y_GRID))
        row.update(compute_s2_observables(z_img, X_GRID, Y_GRID))
        row.update(compute_e2_observables(z_img, X_GRID, Y_GRID))
        if local_idx in expensive_indices:
            row.update(compute_e3_c2_d2_observables(z_img, X_GRID, Y_GRID))
            row.update(compute_minkowski_observables(z_img))
        row.update(compute_multipole_observables(z_img, X_GRID, Y_GRID))
        row.update(compute_renyi_observables(z_img))
        row.update(compute_occupancy_observables(z_img))
        row.update(compute_diagnostic_observables(z_img, X_GRID, Y_GRID))
        rows.append(row)
    observable_df = pd.DataFrame(rows)
    try:
        observable_df.to_parquet(obs_cache_path, index=False)
        print("saved parquet cache:", obs_cache_path)
    except Exception as exc:
        warnings.warn(f"Could not save parquet cache, saving CSV instead: {exc}")
    observable_df.to_csv(obs_cache_csv_path, index=False)
    print("saved CSV cache:", obs_cache_csv_path)

latent_observable_df = latent_mu_df.merge(observable_df, on="event_index", how="inner")
latent_observable_df.to_csv(OBS_TABLE_DIR / "latent_observables_priority1.csv", index=False)
try:
    latent_observable_df.to_parquet(OBS_TABLE_DIR / "latent_observables_priority1.parquet", index=False)
except Exception as exc:
    warnings.warn(f"Could not save latent observable parquet: {exc}")
print("observable columns:", len([c for c in observable_df.columns if c != "event_index"]))
print("merged rows:", len(latent_observable_df))


### Observable Validation

In [ ]:
def validate_priority1_observables(df):
    """Run dataframe-level sanity checks for Priority 1 observables and save a validation table."""
    validation = []
    for col in df.columns:
        if col == "event_index":
            continue
        values = pd.to_numeric(df[col], errors="coerce")
        validation.append({
            "observable_name": col,
            "nan_fraction": float(values.isna().mean()),
            "inf_count": int(np.isinf(values.to_numpy(dtype=float, na_value=np.nan)).sum()),
            "min": float(np.nanmin(values)) if values.notna().any() else np.nan,
            "median": float(np.nanmedian(values)) if values.notna().any() else np.nan,
            "max": float(np.nanmax(values)) if values.notna().any() else np.nan,
        })
    validation_df = pd.DataFrame(validation)
    validation_df.to_csv(OBS_TABLE_DIR / "priority1_observable_validation.csv", index=False)
    if "effective_pixel_multiplicity" in df and "n_active_pixels" in df:
        assert ((df["effective_pixel_multiplicity"] >= 1.0 - 1e-9) | df["effective_pixel_multiplicity"].isna()).all()
        assert ((df["effective_pixel_multiplicity"] <= df["n_active_pixels"] + 1e-9) | df["effective_pixel_multiplicity"].isna()).all()
    if "image_entropy" in df and "n_active_pixels" in df:
        entropy_upper = np.log(df["n_active_pixels"].clip(lower=1))
        assert ((df["image_entropy"] >= -1e-9) | df["image_entropy"].isna()).all()
        assert ((df["image_entropy"] <= entropy_upper + 1e-9) | df["image_entropy"].isna()).all()
    s2_cols = [f"s2_bin_{k:02d}" for k in range(len(s2_edges) - 1) if f"s2_bin_{k:02d}" in df]
    if s2_cols and "s2_total_pair_weight" in df:
        max_s2_diff = float(np.nanmax(np.abs(df[s2_cols].sum(axis=1) - df["s2_total_pair_weight"])))
        print("max |sum S2 bins - pair weight|:", max_s2_diff)
        assert max_s2_diff < 1e-8
    multipole_cols = [c for c in df.columns if c.startswith("multipole_n") and not c.endswith(("cos_phase", "sin_phase"))]
    if multipole_cols:
        max_multipole = float(np.nanmax(df[multipole_cols].to_numpy(dtype=float)))
        print("max multipole magnitude:", max_multipole)
        assert max_multipole <= 1.0 + 1e-8
    print("max image normalization deviation:", float(np.nanmax(df["image_norm_deviation"])))
    return validation_df

validation_df = validate_priority1_observables(observable_df)
display(validation_df.sort_values("nan_fraction", ascending=False).head(12))


### Correlation Utilities and Residualization Against `girth`


In [ ]:
def correlation_matrix(df, row_names, col_names, method="pearson"):
    """Compute a rectangular correlation matrix with pairwise finite-value filtering."""
    out = pd.DataFrame(index=row_names, columns=col_names, dtype=float)
    for row in row_names:
        for col in col_names:
            pair = df[[row, col]].replace([np.inf, -np.inf], np.nan).dropna()
            out.loc[row, col] = pair[row].corr(pair[col], method=method) if len(pair) >= 3 else np.nan
    return out

def save_corr_heatmap(matrix_df, title, path_base, *, vmin=-1.0, vmax=1.0, cmap="coolwarm"):
    """Save a correlation heatmap with adaptive figure dimensions."""
    n_rows, n_cols = matrix_df.shape
    fig_w = max(7.0, min(24.0, 0.38 * n_cols + 3.0))
    fig_h = max(4.0, min(26.0, 0.32 * n_rows + 2.0))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), constrained_layout=True)
    im = ax.imshow(matrix_df.to_numpy(dtype=float), vmin=vmin, vmax=vmax, cmap=cmap, aspect="auto")
    ax.set_xticks(np.arange(n_cols), labels=matrix_df.columns.tolist(), rotation=45, ha="right", fontsize=7)
    ax.set_yticks(np.arange(n_rows), labels=matrix_df.index.tolist(), fontsize=7)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    fig.savefig(path_base.with_suffix(".png"), dpi=220)
    fig.savefig(path_base.with_suffix(".pdf"))
    plt.show()

def residual_suffix(reference_col):
    """Stable residual-column suffix for a conditioning variable name."""
    return "resid_" + re.sub(r"[^0-9A-Za-z]+", "_", str(reference_col)).strip("_")

def residualize_against_reference(df, observable_names, reference_col="girth", n_bins=residualize_n_bins):
    """Residualize observables against a reference observable using binned conditional means.

    For each observable O this returns O - E[O | reference_col]. This is nonlinear and nonparametric at the bin
    level; it is not a simple linear subtraction. The current default reference is girth.
    """
    if reference_col not in df.columns:
        raise KeyError(f"Reference column {reference_col!r} is not in the DataFrame.")
    residuals = pd.DataFrame(index=df.index)
    ref = pd.to_numeric(df[reference_col], errors="coerce")
    valid_ref = ref.replace([np.inf, -np.inf], np.nan).dropna()
    if valid_ref.nunique() < 3:
        raise RuntimeError(f"Not enough finite {reference_col} variation for residualization.")
    bins = pd.qcut(valid_ref, q=min(n_bins, valid_ref.nunique()), duplicates="drop")
    bin_by_index = pd.Series(index=valid_ref.index, data=bins)
    suffix = residual_suffix(reference_col)
    for name in observable_names:
        if name == reference_col:
            continue
        values = pd.to_numeric(df[name], errors="coerce").replace([np.inf, -np.inf], np.nan)
        tmp = pd.DataFrame({"value": values, "bin": bin_by_index})
        conditional_mean = tmp.groupby("bin", observed=True)["value"].transform("mean")
        residuals[f"{name}_{suffix}"] = values - conditional_mean
    return residuals

def residualize_against_girth(df, observable_names, n_bins=residualize_n_bins):
    """Convenience wrapper for the current default residualization against girth."""
    return residualize_against_reference(df, observable_names, reference_col="girth", n_bins=n_bins)

observable_names = [name for name in observable_metadata_df["observable_name"] if name in latent_observable_df.columns]
primary_observable_names = observable_metadata_df.loc[
    (observable_metadata_df["observable_name"].isin(observable_names)) &
    (observable_metadata_df["family"] != "diagnostic") &
    (~observable_metadata_df["observable_name"].str.endswith(("cos_phase", "sin_phase"))),
    "observable_name",
].tolist()
diagnostic_observable_names = observable_metadata_df.loc[
    (observable_metadata_df["observable_name"].isin(observable_names)) &
    (observable_metadata_df["family"] == "diagnostic"),
    "observable_name",
].tolist()

residual_reference_col = "girth"
residual_reference_suffix = residual_suffix(residual_reference_col)
residual_df = residualize_against_reference(latent_observable_df, primary_observable_names, reference_col=residual_reference_col)
residual_df.insert(0, "event_index", latent_observable_df["event_index"].to_numpy())
residual_df.to_csv(OBS_TABLE_DIR / f"observables_residualized_{residual_reference_col}.csv", index=False)
latent_residual_df = pd.concat([latent_observable_df[["event_index"] + latent_correlation_feature_names].reset_index(drop=True), residual_df.drop(columns=["event_index"]).reset_index(drop=True)], axis=1)
print("primary observables:", len(primary_observable_names))
print("diagnostic observables:", len(diagnostic_observable_names))
print("residualized observables:", residual_df.shape[1] - 1)
print("residualized against:", residual_reference_col)


### Family Correlation Outputs

In [ ]:
family_to_observables = defaultdict(list)
for _, row in observable_metadata_df.iterrows():
    name = row["observable_name"]
    if name in latent_observable_df.columns:
        family_to_observables[row["family"]].append(name)

for family, names in family_to_observables.items():
    usable = [n for n in names if pd.to_numeric(latent_observable_df[n], errors="coerce").nunique(dropna=True) > 1]
    if len(usable) < 2:
        continue
    pearson = latent_observable_df[usable].corr(method="pearson")
    spearman = latent_observable_df[usable].corr(method="spearman")
    pearson.to_csv(OBS_TABLE_DIR / f"observable_observable_{family}_pearson.csv")
    spearman.to_csv(OBS_TABLE_DIR / f"observable_observable_{family}_spearman.csv")
    # Keep very large raw blocks in CSV only; heatmaps remain readable for moderate-size families.
    if len(usable) <= 60:
        save_corr_heatmap(pearson, f"Observable-observable Pearson: {family}", OBS_HEATMAP_DIR / f"observable_observable_{family}_pearson")
        save_corr_heatmap(spearman, f"Observable-observable Spearman: {family}", OBS_HEATMAP_DIR / f"observable_observable_{family}_spearman")

def _base_observable_name(name, suffix=None):
    """Map a residualized observable name back to its original metadata name."""
    suffix = suffix or f"_{residual_reference_suffix}"
    return name[: -len(suffix)] if isinstance(name, str) and name.endswith(suffix) else name


def select_top_correlated_columns(corr_df, candidate_cols, top_n=40):
    """Select columns with the largest maximum absolute latent correlation."""
    candidate_cols = [c for c in candidate_cols if c in corr_df.columns]
    if not candidate_cols:
        return []
    scores = corr_df[candidate_cols].abs().replace([np.inf, -np.inf], np.nan).max(axis=0).dropna()
    return scores.sort_values(ascending=False).head(top_n).index.tolist()


def save_family_latent_corr_heatmaps(corr_df, observable_cols, metadata_df, *, path_prefix, title_prefix, top_n=40, residualized=False, residual_suffix_value=None):
    """Save one latent-observable heatmap per observable family using the top correlated variables.

    The complete correlation matrix remains in CSV. These family heatmaps are for readable visualization.
    """
    name_to_family = metadata_df.set_index("observable_name")["family"].to_dict()
    family_to_cols = defaultdict(list)
    for col in observable_cols:
        base = _base_observable_name(col, suffix=residual_suffix_value) if residualized else col
        family = name_to_family.get(base)
        if family is None or family == "diagnostic":
            continue
        family_to_cols[family].append(col)

    rows = []
    for family in sorted(family_to_cols):
        cols = family_to_cols[family]
        top_cols = select_top_correlated_columns(corr_df, cols, top_n=top_n)
        if not top_cols:
            continue
        sub_corr = corr_df[top_cols]
        out_base = OBS_HEATMAP_DIR / f"{path_prefix}_{family}_top{top_n}"
        table_path = OBS_TABLE_DIR / f"{path_prefix}_{family}_top{top_n}.csv"
        sub_corr.to_csv(table_path)
        save_corr_heatmap(
            sub_corr,
            f"{title_prefix}: {family} top {len(top_cols)}",
            out_base,
        )
        rows.append({
            "family": family,
            "n_available": len(cols),
            "n_plotted": len(top_cols),
            "selected_observables": ",".join(top_cols),
            "csv_path": str(table_path),
            "png_path": str(out_base.with_suffix(".png")),
            "pdf_path": str(out_base.with_suffix(".pdf")),
        })
    summary = pd.DataFrame(rows)
    summary.to_csv(OBS_TABLE_DIR / f"{path_prefix}_family_top{top_n}_summary.csv", index=False)
    return summary

latent_raw_corr = correlation_matrix(latent_observable_df, latent_correlation_feature_names, primary_observable_names, method="pearson")
latent_raw_corr.to_csv(OBS_TABLE_DIR / "latent_observable_corr_raw.csv")
raw_family_heatmap_summary = save_family_latent_corr_heatmaps(
    latent_raw_corr,
    primary_observable_names,
    observable_metadata_df,
    path_prefix="latent_observable_corr_raw",
    title_prefix="Latent mu plus sigma_18 vs raw observables",
    top_n=40,
    residualized=False,
)

residual_names = [c for c in residual_df.columns if c != "event_index"]
latent_resid_corr = correlation_matrix(latent_residual_df, latent_correlation_feature_names, residual_names, method="pearson")
latent_resid_corr.to_csv(OBS_TABLE_DIR / f"latent_observable_corr_residualized_{residual_reference_col}.csv")
resid_family_heatmap_summary = save_family_latent_corr_heatmaps(
    latent_resid_corr,
    residual_names,
    observable_metadata_df,
    path_prefix=f"latent_observable_corr_residualized_{residual_reference_col}",
    title_prefix=f"Latent mu plus sigma_18 vs observables residualized against {residual_reference_col}",
    top_n=40,
    residualized=True,
)

latent_diag_corr = correlation_matrix(latent_observable_df, latent_correlation_feature_names, diagnostic_observable_names, method="pearson")
latent_diag_corr.to_csv(OBS_TABLE_DIR / "latent_diagnostic_corr_raw.csv")
if 1 <= len(diagnostic_observable_names) <= 80:
    save_corr_heatmap(latent_diag_corr, "Latent mu plus sigma_18 vs diagnostic variables", OBS_HEATMAP_DIR / "latent_diagnostic_corr_raw")

print("Raw family heatmaps saved:")
display(raw_family_heatmap_summary[["family", "n_available", "n_plotted", "selected_observables"]])
print(f"Residualized-against-{residual_reference_col} family heatmaps saved:")
display(resid_family_heatmap_summary[["family", "n_available", "n_plotted", "selected_observables"]])


### Priority 1 Summary

In [ ]:
family_counts = observable_metadata_df[observable_metadata_df["observable_name"].isin(observable_df.columns)].groupby("family")["observable_name"].nunique().sort_values(ascending=False)
nan_summary = validation_df.sort_values("nan_fraction", ascending=False)[["observable_name", "nan_fraction"]].head(15)
raw_long = latent_raw_corr.abs().stack().sort_values(ascending=False).head(20).reset_index()
raw_long.columns = ["latent", "observable", "abs_corr"]
resid_long = latent_resid_corr.abs().stack().sort_values(ascending=False).head(20).reset_index()
resid_long.columns = ["latent", "observable", "abs_corr"]
runtime_seconds = time.perf_counter() - obs_start_time
summary_payload = {
    "family_counts": family_counts.to_dict(),
    "successful_jets": int(latent_observable_df["image_valid"].sum()) if "image_valid" in latent_observable_df else int(len(latent_observable_df)),
    "total_jets": int(len(latent_observable_df)),
    "runtime_seconds": float(runtime_seconds),
    "residualized_against": residual_reference_col,
    "cache_locations": {
        "latent_mu": str(latent_cache_path),
        "images": str(image_cache_path),
        "observables_parquet": str(obs_cache_path),
        "observables_csv": str(obs_cache_csv_path),
    },
    "configuration_hash": observable_config_hash,
}
(OBS_TABLE_DIR / "priority1_summary.json").write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary_payload, indent=2, ensure_ascii=False))
print("\nObservable count by family:")
display(family_counts.to_frame("n_observables"))
print("\nLargest NaN fractions:")
display(nan_summary)
print("\nStrongest raw latent-observable correlations:")
display(raw_long)
print(f"\nStrongest latent-observable correlations after residualizing against {residual_reference_col}:")
display(resid_long)
print("\nSaved observable metadata:", OBS_TABLE_DIR / "observable_metadata.csv")
print("Saved raw correlations:", OBS_TABLE_DIR / "latent_observable_corr_raw.csv")
print("Saved residualized correlations:", OBS_TABLE_DIR / f"latent_observable_corr_residualized_{residual_reference_col}.csv")


## Priority 2 Shower and Texture Extension

This section adds the Priority 2 observables on top of the already computed Priority 1 table. It is intentionally independent of the Priority 1 cache: rerunning these cells does not invalidate the existing Priority 1 outputs.

Implemented here:

- Cambridge/Aachen-style pixel reclustering fallback and hardest-branch Lund declustering;
- exclusive 2-subjet and 3-subjet observables from the same clustering tree;
- dimensionless splitting-scale summaries;
- Lund-plane histogram PCA;
- Haar wavelet texture observables;
- Fourier power-spectrum observables.

`fastjet` is detected if available, but the default implementation below uses a deterministic pixel-level fallback so the notebook can run without optional external jet-clustering bindings.

In [ ]:
### Priority 2 Configuration

try:
    import fastjet as fj
    FASTJET_AVAILABLE = True
except Exception as exc:
    fj = None
    FASTJET_AVAILABLE = False
    warnings.warn(f"fastjet is not available; using the deterministic pixel C/A fallback. Reason: {exc}")

try:
    import pywt
    PYWT_AVAILABLE = True
except Exception as exc:
    pywt = None
    PYWT_AVAILABLE = False
    warnings.warn(f"pywt is not available; using the built-in Haar wavelet fallback. Reason: {exc}")

priority2_cache_version = "priority2_shower_texture_v1"

# Shower modules are more expensive than wavelet/Fourier texture. If the sample is larger than this,
# Lund/subjet observables are computed on a reproducible subset and left as NaN for the other jets.
priority2_shower_max_jets = min(int(expensive_max_jets), 10000)
max_lund_pixels = 35
lund_pixel_threshold = max(pixel_active_threshold, 1e-8)
enable_fastjet_lund = False  # fallback is used by default to avoid a hard dependency on fastjet bindings.

lund_z_cut = 0.05
lund_delta_cut = 0.2
lund_kt_cut = 0.01
lund_log_inv_delta_edges = np.linspace(-np.log(2.0), 5.0, 9)
lund_log_inv_kt_edges = np.linspace(-np.log(2.0), 7.0, 9)
lund_pc_count = 10

wavelet_name = "haar"
wavelet_levels = 3
fft_radial_bins = 10
fft_angular_harmonics = [2, 3, 4]
fourier_eps = 1e-30

priority2_config = {
    "base_observable_config_hash": observable_config_hash,
    "priority2_cache_version": priority2_cache_version,
    "priority2_shower_max_jets": priority2_shower_max_jets,
    "max_lund_pixels": max_lund_pixels,
    "lund_pixel_threshold": lund_pixel_threshold,
    "enable_fastjet_lund": enable_fastjet_lund,
    "lund_z_cut": lund_z_cut,
    "lund_delta_cut": lund_delta_cut,
    "lund_kt_cut": lund_kt_cut,
    "lund_log_inv_delta_edges": lund_log_inv_delta_edges.tolist(),
    "lund_log_inv_kt_edges": lund_log_inv_kt_edges.tolist(),
    "lund_pc_count": lund_pc_count,
    "wavelet_name": wavelet_name,
    "wavelet_levels": wavelet_levels,
    "fft_radial_bins": fft_radial_bins,
    "fft_angular_harmonics": fft_angular_harmonics,
}
priority2_config_hash = hashlib.sha256(json.dumps(priority2_config, sort_keys=True).encode("utf-8")).hexdigest()[:16]
print("fastjet available:", FASTJET_AVAILABLE)
print("pywt available:", PYWT_AVAILABLE)
print("Priority 2 config hash:", priority2_config_hash)
print("Priority 2 shower max jets:", priority2_shower_max_jets)
print("Priority 2 max Lund pixels:", max_lund_pixels)

In [ ]:
### Priority 2 Observable Functions

def _empty_lund_summary():
    """Return NaN/zero defaults for C/A-Lund observables when no splitting is available."""
    out = {
        "lund_n_splits": 0,
        "lund_n_hard": 0,
        "lund_n_soft": 0,
        "lund_n_wide": 0,
        "lund_n_collinear": 0,
        "lund_max_kt_rel": np.nan,
        "lund_mean_kt_rel": np.nan,
        "lund_hardest_z": np.nan,
        "lund_hardest_delta": np.nan,
        "lund_first_z": np.nan,
        "lund_first_delta": np.nan,
        "lund_tree_depth": 0,
        "kt_split_12": np.nan,
        "kt_split_23": np.nan,
        "kt_split_34": np.nan,
        "subjet2_z1": np.nan,
        "subjet2_z2": np.nan,
        "subjet2_z2_over_z1": np.nan,
        "subjet2_delta12": np.nan,
        "subjet2_mass_dimless": np.nan,
        "subjet3_z1": np.nan,
        "subjet3_z2": np.nan,
        "subjet3_z3": np.nan,
        "subjet3_d12": np.nan,
        "subjet3_d13": np.nan,
        "subjet3_d23": np.nan,
        "subjet3_min_delta": np.nan,
        "subjet3_max_delta": np.nan,
        "subjet3_mean_delta": np.nan,
        "lund_selected_energy_fraction": np.nan,
        "lund_n_selected_pixels": 0,
    }
    return out


def lund_hist_column_names():
    """Stable flattened Lund-plane histogram column names."""
    cols = []
    for i in range(len(lund_log_inv_delta_edges) - 1):
        for j in range(len(lund_log_inv_kt_edges) - 1):
            cols.append(f"lund_hist_d{i:02d}_kt{j:02d}")
    return cols


def empty_lund_hist_row():
    """Return a zero-valued Lund histogram row."""
    return {name: 0.0 for name in lund_hist_column_names()}


def build_ca_tree_fallback(z_img, x_grid, y_grid):
    """Build a Cambridge/Aachen-style binary tree from active image pixels.

    Input shape is a normalized 2D image plus matching normalized coordinate grids. The fallback treats pixels as
    massless pseudo-constituents with relative transverse momenta z_a and greedily merges the closest pair in
    normalized angular distance. The returned tree is pixelization-sensitive and uses at most max_lund_pixels.
    """
    z, x, y = select_active_pixels(z_img, x_grid, y_grid, threshold=lund_pixel_threshold, max_pixels=max_lund_pixels)
    n = len(z)
    if n == 0:
        return {}, None, z, x, y
    nodes = {}
    active = []
    for i in range(n):
        nodes[i] = {"id": i, "left": None, "right": None, "pt": float(z[i]), "x": float(x[i]), "y": float(y[i]), "depth": 0}
        active.append(i)
    next_id = n
    while len(active) > 1:
        best = None
        best_d2 = np.inf
        for ia in range(len(active) - 1):
            na = nodes[active[ia]]
            for ib in range(ia + 1, len(active)):
                nb = nodes[active[ib]]
                d2 = (na["x"] - nb["x"]) ** 2 + (na["y"] - nb["y"]) ** 2
                if d2 < best_d2:
                    best_d2 = d2
                    best = (ia, ib)
        ia, ib = best
        id_a = active[ia]
        id_b = active[ib]
        na = nodes[id_a]
        nb = nodes[id_b]
        pt = float(na["pt"] + nb["pt"])
        if pt > 0:
            x_new = float((na["pt"] * na["x"] + nb["pt"] * nb["x"]) / pt)
            y_new = float((na["pt"] * na["y"] + nb["pt"] * nb["y"]) / pt)
        else:
            x_new = 0.5 * (na["x"] + nb["x"])
            y_new = 0.5 * (na["y"] + nb["y"])
        nodes[next_id] = {
            "id": next_id,
            "left": id_a,
            "right": id_b,
            "pt": pt,
            "x": x_new,
            "y": y_new,
            "depth": 1 + max(int(na["depth"]), int(nb["depth"])),
        }
        for idx in sorted(best, reverse=True):
            active.pop(idx)
        active.append(next_id)
        next_id += 1
    return nodes, active[0], z, x, y


def hardest_branch_splits(nodes, root_id):
    """Decluster a fallback C/A tree along the hardest branch and return splitting records."""
    splits = []
    current = root_id
    while current is not None:
        node = nodes[current]
        left = node["left"]
        right = node["right"]
        if left is None or right is None:
            break
        a = nodes[left]
        b = nodes[right]
        denom = float(a["pt"] + b["pt"])
        delta = float(math.hypot(a["x"] - b["x"], a["y"] - b["y"]))
        zsplit = float(min(a["pt"], b["pt"]) / denom) if denom > 0 else np.nan
        kt_rel = float(zsplit * delta) if np.isfinite(zsplit) else np.nan
        splits.append({
            "delta": delta,
            "z": zsplit,
            "kt_rel": kt_rel,
            "pt_hard": float(max(a["pt"], b["pt"])),
            "pt_soft": float(min(a["pt"], b["pt"])),
        })
        current = left if a["pt"] >= b["pt"] else right
    return splits


def exclusive_subjets_from_tree(nodes, root_id, n_subjets):
    """Return exclusive subjet nodes by recursively splitting the highest-pt composite node."""
    if root_id is None:
        return []
    clusters = [root_id]
    while len(clusters) < n_subjets:
        composite = [cid for cid in clusters if nodes[cid]["left"] is not None and nodes[cid]["right"] is not None]
        if not composite:
            break
        split_id = max(composite, key=lambda cid: nodes[cid]["pt"])
        clusters.remove(split_id)
        clusters.extend([nodes[split_id]["left"], nodes[split_id]["right"]])
    return sorted([nodes[cid] for cid in clusters], key=lambda item: item["pt"], reverse=True)


def compute_lund_histogram_from_splits(splits):
    """Compute a flattened 8x8 Lund-plane histogram from hardest-branch splittings."""
    hist = np.zeros((len(lund_log_inv_delta_edges) - 1, len(lund_log_inv_kt_edges) - 1), dtype=np.float64)
    for split in splits:
        delta = split["delta"]
        kt_rel = split["kt_rel"]
        if not np.isfinite(delta) or not np.isfinite(kt_rel) or delta <= 0 or kt_rel <= 0:
            continue
        xval = math.log(1.0 / delta)
        yval = math.log(1.0 / kt_rel)
        i = np.searchsorted(lund_log_inv_delta_edges, xval, side="right") - 1
        j = np.searchsorted(lund_log_inv_kt_edges, yval, side="right") - 1
        if 0 <= i < hist.shape[0] and 0 <= j < hist.shape[1]:
            hist[i, j] += 1.0
    return {name: float(value) for name, value in zip(lund_hist_column_names(), hist.ravel())}


def compute_shower_observables(z_img, x_grid, y_grid):
    """Compute Priority 2 shower/clustering-history observables from a normalized 2D image.

    The default path uses the deterministic C/A-like fallback tree. All pT-like quantities are dimensionless
    fractions of the normalized image pT; kt_rel is z*Delta with Delta measured in units of AK4 radius R.
    """
    nodes, root_id, z, x, y = build_ca_tree_fallback(z_img, x_grid, y_grid)
    out = _empty_lund_summary()
    out["lund_selected_energy_fraction"] = float(np.sum(z))
    out["lund_n_selected_pixels"] = int(len(z))
    if root_id is None or len(z) < 2:
        return out, empty_lund_hist_row()
    splits = hardest_branch_splits(nodes, root_id)
    out["lund_tree_depth"] = int(nodes[root_id]["depth"])
    out["lund_n_splits"] = int(len(splits))
    if splits:
        zvals = np.array([s["z"] for s in splits], dtype=np.float64)
        deltas = np.array([s["delta"] for s in splits], dtype=np.float64)
        ktvals = np.array([s["kt_rel"] for s in splits], dtype=np.float64)
        out.update({
            "lund_n_hard": int(np.count_nonzero(zvals >= lund_z_cut)),
            "lund_n_soft": int(np.count_nonzero(zvals < lund_z_cut)),
            "lund_n_wide": int(np.count_nonzero(deltas >= lund_delta_cut)),
            "lund_n_collinear": int(np.count_nonzero(deltas < lund_delta_cut)),
            "lund_max_kt_rel": float(np.nanmax(ktvals)),
            "lund_mean_kt_rel": float(np.nanmean(ktvals)),
            "lund_first_z": float(zvals[0]),
            "lund_first_delta": float(deltas[0]),
        })
        hardest_idx = int(np.nanargmax(ktvals))
        out["lund_hardest_z"] = float(zvals[hardest_idx])
        out["lund_hardest_delta"] = float(deltas[hardest_idx])
        kt_sorted = np.sort(ktvals[np.isfinite(ktvals)])[::-1]
        for idx, name in enumerate(["kt_split_12", "kt_split_23", "kt_split_34"]):
            out[name] = float(kt_sorted[idx]) if idx < len(kt_sorted) else np.nan
    sub2 = exclusive_subjets_from_tree(nodes, root_id, 2)
    if len(sub2) >= 2:
        z1, z2 = float(sub2[0]["pt"]), float(sub2[1]["pt"])
        d12 = float(math.hypot(sub2[0]["x"] - sub2[1]["x"], sub2[0]["y"] - sub2[1]["y"]))
        out["subjet2_z1"] = z1
        out["subjet2_z2"] = z2
        out["subjet2_z2_over_z1"] = float(z2 / z1) if z1 > 0 else np.nan
        out["subjet2_delta12"] = d12
        out["subjet2_mass_dimless"] = float(math.sqrt(max(z1 * z2, 0.0)) * d12)
    sub3 = exclusive_subjets_from_tree(nodes, root_id, 3)
    if len(sub3) >= 3:
        pts = [float(s["pt"]) for s in sub3[:3]]
        ds = [
            float(math.hypot(sub3[0]["x"] - sub3[1]["x"], sub3[0]["y"] - sub3[1]["y"])),
            float(math.hypot(sub3[0]["x"] - sub3[2]["x"], sub3[0]["y"] - sub3[2]["y"])),
            float(math.hypot(sub3[1]["x"] - sub3[2]["x"], sub3[1]["y"] - sub3[2]["y"])),
        ]
        out.update({
            "subjet3_z1": pts[0],
            "subjet3_z2": pts[1],
            "subjet3_z3": pts[2],
            "subjet3_d12": ds[0],
            "subjet3_d13": ds[1],
            "subjet3_d23": ds[2],
            "subjet3_min_delta": float(np.min(ds)),
            "subjet3_max_delta": float(np.max(ds)),
            "subjet3_mean_delta": float(np.mean(ds)),
        })
    return out, compute_lund_histogram_from_splits(splits)


def _safe_kurtosis(values):
    """Return non-excess kurtosis with NaN for constant arrays."""
    arr = np.asarray(values, dtype=np.float64).ravel()
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan
    mean = float(np.mean(arr))
    std = float(np.std(arr))
    if std <= 0:
        return np.nan
    return float(np.mean(((arr - mean) / std) ** 4))


def _coeff_stats(prefix, coeffs):
    """Basic coefficient-energy and shape statistics for one wavelet detail block."""
    arr = np.asarray(coeffs, dtype=np.float64).ravel()
    if arr.size == 0:
        return {
            f"{prefix}_energy": np.nan,
            f"{prefix}_mean_abs": np.nan,
            f"{prefix}_max_abs": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_kurtosis": np.nan,
        }
    abs_arr = np.abs(arr)
    return {
        f"{prefix}_energy": float(np.sum(arr ** 2)),
        f"{prefix}_mean_abs": float(np.mean(abs_arr)),
        f"{prefix}_max_abs": float(np.max(abs_arr)),
        f"{prefix}_std": float(np.std(arr)),
        f"{prefix}_kurtosis": _safe_kurtosis(arr),
    }


def _manual_haar_details(z_img, levels):
    """Built-in 2D Haar fallback returning detail coefficients per level."""
    current = np.asarray(z_img, dtype=np.float64)
    details = []
    for _level in range(int(levels)):
        h, w = current.shape
        h2 = (h // 2) * 2
        w2 = (w // 2) * 2
        if h2 < 2 or w2 < 2:
            break
        trimmed = current[:h2, :w2]
        a = trimmed[0::2, 0::2]
        b = trimmed[0::2, 1::2]
        c = trimmed[1::2, 0::2]
        d = trimmed[1::2, 1::2]
        approx = 0.5 * (a + b + c + d)
        horizontal = 0.5 * (a + b - c - d)
        vertical = 0.5 * (a - b + c - d)
        diagonal = 0.5 * (a - b - c + d)
        details.append((horizontal, vertical, diagonal))
        current = approx
    return details


def compute_wavelet_observables(z_img):
    """Compute multiscale 2D wavelet texture observables from the image array.

    Uses pywt when available; otherwise uses an equivalent built-in Haar fallback. These observables are
    pixelization-sensitive and should be interpreted as image-texture diagnostics rather than IRC-safe quantities.
    """
    if PYWT_AVAILABLE:
        coeffs = pywt.wavedec2(np.asarray(z_img, dtype=np.float64), wavelet=wavelet_name, level=wavelet_levels, mode="periodization")
        details = list(reversed(coeffs[1:]))
    else:
        details = _manual_haar_details(z_img, wavelet_levels)
    out = {}
    total_h = 0.0
    total_v = 0.0
    total_d = 0.0
    level_energies = []
    for level in range(1, wavelet_levels + 1):
        if level <= len(details):
            h_coeff, v_coeff, d_coeff = details[level - 1]
        else:
            h_coeff = v_coeff = d_coeff = np.array([], dtype=np.float64)
        for direction, coeff in [("horizontal", h_coeff), ("vertical", v_coeff), ("diagonal", d_coeff)]:
            out.update(_coeff_stats(f"wavelet_L{level}_{direction}", coeff))
        e_h = float(np.nansum(np.asarray(h_coeff, dtype=np.float64) ** 2))
        e_v = float(np.nansum(np.asarray(v_coeff, dtype=np.float64) ** 2))
        e_d = float(np.nansum(np.asarray(d_coeff, dtype=np.float64) ** 2))
        total_h += e_h
        total_v += e_v
        total_d += e_d
        level_energies.append(e_h + e_v + e_d)
    fine = level_energies[0] if level_energies else np.nan
    coarse = level_energies[-1] if level_energies else np.nan
    total_detail = total_h + total_v + total_d
    out["wavelet_fine_over_coarse_energy"] = float(fine / coarse) if np.isfinite(fine) and np.isfinite(coarse) and coarse > 0 else np.nan
    out["wavelet_horizontal_over_vertical_energy"] = float(total_h / total_v) if total_v > 0 else np.nan
    out["wavelet_diagonal_over_total_detail_energy"] = float(total_d / total_detail) if total_detail > 0 else np.nan
    return out


def compute_fourier_observables(z_img):
    """Compute Fourier radial power fractions and angular power harmonics with the DC component removed."""
    arr = np.asarray(z_img, dtype=np.float64)
    h, w = arr.shape
    power = np.abs(np.fft.fft2(arr)) ** 2
    power[0, 0] = 0.0
    fy = np.fft.fftfreq(h, d=1.0)
    fx = np.fft.fftfreq(w, d=1.0)
    fx_grid, fy_grid = np.meshgrid(fx, fy, indexing="xy")
    radius = np.sqrt(fx_grid ** 2 + fy_grid ** 2)
    theta = np.arctan2(fy_grid, fx_grid)
    total = float(np.sum(power))
    edges = np.linspace(0.0, float(radius.max()) + np.finfo(np.float64).eps, fft_radial_bins + 1)
    out = {}
    for i in range(fft_radial_bins):
        mask = (radius >= edges[i]) & (radius < edges[i + 1])
        out[f"fft_radial_bin_{i:02d}"] = float(np.sum(power[mask]) / total) if total > 0 else np.nan
    for nval in fft_angular_harmonics:
        moment = np.sum(power * np.exp(1j * nval * theta))
        out[f"fft_angular_harmonic_n{nval}"] = float(np.abs(moment) / total) if total > 0 else np.nan
    low_mask = (radius >= edges[1]) & (radius < edges[max(2, fft_radial_bins // 3 + 1)])
    high_mask = radius >= edges[max(1, 2 * fft_radial_bins // 3)]
    low = float(np.sum(power[low_mask]))
    high = float(np.sum(power[high_mask]))
    out["fft_low_freq_frac"] = float(low / total) if total > 0 else np.nan
    out["fft_high_freq_frac"] = float(high / total) if total > 0 else np.nan
    out["fft_high_low_ratio"] = float(high / max(low, fourier_eps)) if total > 0 and low > 0 else np.nan
    return out


def compute_pca_block(df, feature_cols, *, n_components, prefix, event_col="event_index"):
    """Standardize a feature block, remove near-zero variance columns, and return PCA scores via SVD."""
    feature_cols = [c for c in feature_cols if c in df.columns]
    pc_names = [f"{prefix}_pc{i:02d}" for i in range(1, int(n_components) + 1)]
    pc_df = pd.DataFrame({event_col: df[event_col].to_numpy()})
    for name in pc_names:
        pc_df[name] = np.nan
    meta_rows = []
    if not feature_cols:
        return pc_df, pd.DataFrame(meta_rows)
    X = df[feature_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
    valid_rows = np.isfinite(X).any(axis=1)
    if valid_rows.sum() < 3:
        return pc_df, pd.DataFrame(meta_rows)
    X_valid = X[valid_rows]
    means = np.nanmean(X_valid, axis=0)
    all_nan = ~np.isfinite(means)
    means[all_nan] = 0.0
    X_filled = np.where(np.isfinite(X_valid), X_valid, means)
    stds = np.std(X_filled, axis=0)
    keep = np.isfinite(stds) & (stds > 1e-12) & (~all_nan)
    if keep.sum() == 0:
        return pc_df, pd.DataFrame(meta_rows)
    Z = (X_filled[:, keep] - means[keep]) / stds[keep]
    U, S, Vt = np.linalg.svd(Z, full_matrices=False)
    n_keep = min(int(n_components), U.shape[1])
    scores = U[:, :n_keep] * S[:n_keep]
    valid_index = np.flatnonzero(valid_rows)
    for i in range(n_keep):
        pc_df.loc[valid_index, pc_names[i]] = scores[:, i]
    variances = (S ** 2) / max(Z.shape[0] - 1, 1)
    total_var = float(np.sum(variances))
    kept_cols = np.array(feature_cols, dtype=object)[keep].tolist()
    for i in range(n_keep):
        loading = Vt[i]
        strongest = np.argsort(np.abs(loading))[::-1][:10]
        meta_rows.append({
            "component": pc_names[i],
            "explained_variance": float(variances[i]),
            "explained_variance_ratio": float(variances[i] / total_var) if total_var > 0 else np.nan,
            "n_input_features": int(len(feature_cols)),
            "n_kept_features": int(len(kept_cols)),
            "strongest_input_features": ",".join([kept_cols[j] for j in strongest]),
        })
    return pc_df, pd.DataFrame(meta_rows)


def run_priority2_sanity_tests():
    """Run lightweight synthetic-image checks for the Priority 2 modules."""
    center = synthetic_image([(obs_img_size // 2, obs_img_size // 2, 1.0)])
    two = synthetic_image([(obs_img_size // 2, obs_img_size // 2 - 4, 0.5), (obs_img_size // 2, obs_img_size // 2 + 4, 0.5)])
    center_prob, _ = prepare_probability_image(center)
    two_prob, _ = prepare_probability_image(two)
    center_shower, center_hist = compute_shower_observables(center_prob, X_GRID, Y_GRID)
    two_shower, two_hist = compute_shower_observables(two_prob, X_GRID, Y_GRID)
    assert center_shower["lund_n_splits"] == 0
    assert two_shower["lund_n_splits"] == 1
    assert np.isclose(two_shower["subjet2_z1"], 0.5)
    assert np.isclose(two_shower["subjet2_z2"], 0.5)
    assert np.isclose(sum(two_hist.values()), 1.0)
    wave = compute_wavelet_observables(two_prob)
    fft = compute_fourier_observables(two_prob)
    assert any(np.isfinite(v) for v in wave.values())
    assert 0.0 <= fft["fft_low_freq_frac"] <= 1.0
    assert 0.0 <= fft["fft_high_freq_frac"] <= 1.0
    return pd.DataFrame([
        {"synthetic_image": "center_pixel", **center_shower},
        {"synthetic_image": "two_equal_pixels", **two_shower},
    ])


priority2_sanity_df = run_priority2_sanity_tests()
display(priority2_sanity_df[["synthetic_image", "lund_n_splits", "subjet2_z1", "subjet2_z2", "subjet2_delta12"]])

In [ ]:
### Compute and Cache Priority 2 Observables

priority2_metadata_rows = []
for name in [
    "lund_n_splits", "lund_n_hard", "lund_n_soft", "lund_n_wide", "lund_n_collinear",
    "lund_max_kt_rel", "lund_mean_kt_rel", "lund_hardest_z", "lund_hardest_delta",
    "lund_first_z", "lund_first_delta", "lund_tree_depth",
]:
    add_metadata(priority2_metadata_rows, name, "shower", "lund_declustering", f"C/A hardest-branch Lund observable {name}", expensive=True, parameters={"z_cut": lund_z_cut, "delta_cut": lund_delta_cut, "kt_cut": lund_kt_cut, "max_lund_pixels": max_lund_pixels})
for name in ["kt_split_12", "kt_split_23", "kt_split_34"]:
    add_metadata(priority2_metadata_rows, name, "shower", "kt_splitting", f"Dimensionless fallback kT splitting scale {name}", expensive=True, parameters={"max_lund_pixels": max_lund_pixels})
for name in ["subjet2_z1", "subjet2_z2", "subjet2_z2_over_z1", "subjet2_delta12", "subjet2_mass_dimless"]:
    add_metadata(priority2_metadata_rows, name, "shower", "exclusive_2subjet", f"Exclusive 2-subjet observable {name}", expensive=True, parameters={"max_lund_pixels": max_lund_pixels})
for name in ["subjet3_z1", "subjet3_z2", "subjet3_z3", "subjet3_d12", "subjet3_d13", "subjet3_d23", "subjet3_min_delta", "subjet3_max_delta", "subjet3_mean_delta"]:
    add_metadata(priority2_metadata_rows, name, "shower", "exclusive_3subjet", f"Exclusive 3-subjet observable {name}", expensive=True, parameters={"max_lund_pixels": max_lund_pixels})
for i in range(1, lund_pc_count + 1):
    add_metadata(priority2_metadata_rows, f"lund_pc{i:02d}", "shower", "lund_histogram_pca", "PCA component of flattened Lund-plane histograms", expensive=True, parameters={"n_components": lund_pc_count, "max_lund_pixels": max_lund_pixels})
for name in ["lund_selected_energy_fraction", "lund_n_selected_pixels"]:
    add_metadata(priority2_metadata_rows, name, "diagnostic", "priority2_approximation", f"Priority 2 approximation diagnostic {name}", irc_safe=False, pixelization_sensitive=True, expensive=True)

for level in range(1, wavelet_levels + 1):
    for direction in ["horizontal", "vertical", "diagonal"]:
        for stat in ["energy", "mean_abs", "max_abs", "std", "kurtosis"]:
            add_metadata(priority2_metadata_rows, f"wavelet_L{level}_{direction}_{stat}", "texture", "wavelet", "2D wavelet texture statistic", irc_safe=False, pixelization_sensitive=True, parameters={"wavelet": wavelet_name, "level": level, "direction": direction})
for name in ["wavelet_fine_over_coarse_energy", "wavelet_horizontal_over_vertical_energy", "wavelet_diagonal_over_total_detail_energy"]:
    add_metadata(priority2_metadata_rows, name, "texture", "wavelet_ratio", "Wavelet detail-energy ratio", irc_safe=False, pixelization_sensitive=True, parameters={"wavelet": wavelet_name, "levels": wavelet_levels})
for i in range(fft_radial_bins):
    add_metadata(priority2_metadata_rows, f"fft_radial_bin_{i:02d}", "texture", "fourier_radial", "DC-removed radial Fourier power fraction", irc_safe=False, pixelization_sensitive=True, parameters={"bin": i, "n_bins": fft_radial_bins})
for nval in fft_angular_harmonics:
    add_metadata(priority2_metadata_rows, f"fft_angular_harmonic_n{nval}", "texture", "fourier_angular", "DC-removed angular Fourier power harmonic", irc_safe=False, rotation_invariant=True, pixelization_sensitive=True, parameters={"n": nval})
for name in ["fft_low_freq_frac", "fft_high_freq_frac", "fft_high_low_ratio"]:
    add_metadata(priority2_metadata_rows, name, "texture", "fourier_summary", "DC-removed Fourier power summary", irc_safe=False, pixelization_sensitive=True)

priority2_metadata_df = pd.DataFrame(priority2_metadata_rows).drop_duplicates("observable_name")
observable_metadata_df = pd.concat([observable_metadata_df, priority2_metadata_df], ignore_index=True).drop_duplicates("observable_name", keep="last")
observable_metadata_df.to_csv(OBS_TABLE_DIR / "observable_metadata.csv", index=False)

priority2_obs_cache_path = OBS_CACHE_DIR / f"priority2_observables_{priority2_config_hash}.parquet"
priority2_obs_cache_csv_path = OBS_CACHE_DIR / f"priority2_observables_{priority2_config_hash}.csv"
priority2_lund_hist_cache_path = OBS_CACHE_DIR / f"priority2_lund_histograms_{priority2_config_hash}.parquet"
priority2_lund_hist_cache_csv_path = OBS_CACHE_DIR / f"priority2_lund_histograms_{priority2_config_hash}.csv"

if priority2_obs_cache_path.exists():
    priority2_observable_df = pd.read_parquet(priority2_obs_cache_path)
    print("loaded Priority 2 observables from parquet cache:", priority2_obs_cache_path)
    if priority2_lund_hist_cache_path.exists():
        lund_histogram_df = pd.read_parquet(priority2_lund_hist_cache_path)
    elif priority2_lund_hist_cache_csv_path.exists():
        lund_histogram_df = pd.read_csv(priority2_lund_hist_cache_csv_path)
    else:
        lund_histogram_df = pd.DataFrame({"event_index": priority2_observable_df["event_index"]})
elif priority2_obs_cache_csv_path.exists():
    priority2_observable_df = pd.read_csv(priority2_obs_cache_csv_path)
    print("loaded Priority 2 observables from CSV cache:", priority2_obs_cache_csv_path)
    if priority2_lund_hist_cache_path.exists():
        lund_histogram_df = pd.read_parquet(priority2_lund_hist_cache_path)
    elif priority2_lund_hist_cache_csv_path.exists():
        lund_histogram_df = pd.read_csv(priority2_lund_hist_cache_csv_path)
    else:
        lund_histogram_df = pd.DataFrame({"event_index": priority2_observable_df["event_index"]})
else:
    if len(observable_event_indices) > priority2_shower_max_jets:
        shower_indices = set(rng.choice(np.arange(len(observable_event_indices)), size=int(priority2_shower_max_jets), replace=False).tolist())
        print(f"Priority 2 shower modules run on {priority2_shower_max_jets} reproducible jets; other rows receive NaN for shower columns")
    else:
        shower_indices = set(range(len(observable_event_indices)))

    rows = []
    hist_rows = []
    for local_idx, event_index in tqdm(list(enumerate(observable_event_indices)), desc="Compute Priority 2 observables"):
        z_img, prep = prepare_probability_image(normalized_images[local_idx])
        row = {"event_index": int(event_index)}
        hist_row = {"event_index": int(event_index)}
        if prep["valid"]:
            row.update(compute_wavelet_observables(z_img))
            row.update(compute_fourier_observables(z_img))
            if local_idx in shower_indices:
                shower_obs, hist_obs = compute_shower_observables(z_img, X_GRID, Y_GRID)
                row.update(shower_obs)
                hist_row.update(hist_obs)
        rows.append(row)
        hist_rows.append(hist_row)

    priority2_raw_observable_df = pd.DataFrame(rows)
    lund_histogram_df = pd.DataFrame(hist_rows)
    lund_hist_cols = [c for c in lund_hist_column_names() if c in lund_histogram_df.columns]
    lund_pc_df, lund_pca_metadata_df = compute_pca_block(lund_histogram_df, lund_hist_cols, n_components=lund_pc_count, prefix="lund")
    lund_pca_metadata_df.to_csv(OBS_TABLE_DIR / "priority2_lund_pca_metadata.csv", index=False)
    priority2_observable_df = priority2_raw_observable_df.merge(lund_pc_df, on="event_index", how="left")

    try:
        priority2_observable_df.to_parquet(priority2_obs_cache_path, index=False)
        lund_histogram_df.to_parquet(priority2_lund_hist_cache_path, index=False)
        print("saved Priority 2 parquet caches")
    except Exception as exc:
        warnings.warn(f"Could not save Priority 2 parquet cache, saving CSV instead: {exc}")
    priority2_observable_df.to_csv(priority2_obs_cache_csv_path, index=False)
    lund_histogram_df.to_csv(priority2_lund_hist_cache_csv_path, index=False)
    print("saved Priority 2 CSV caches:", priority2_obs_cache_csv_path, priority2_lund_hist_cache_csv_path)

priority2_cols = [c for c in priority2_observable_df.columns if c != "event_index"]
observable_df = observable_df.drop(columns=[c for c in priority2_cols if c in observable_df.columns], errors="ignore")
observable_df = observable_df.merge(priority2_observable_df, on="event_index", how="left")
latent_observable_df = latent_mu_df.merge(observable_df, on="event_index", how="inner")
latent_observable_df.to_csv(OBS_TABLE_DIR / "latent_observables_priority1_plus_priority2.csv", index=False)
try:
    latent_observable_df.to_parquet(OBS_TABLE_DIR / "latent_observables_priority1_plus_priority2.parquet", index=False)
except Exception as exc:
    warnings.warn(f"Could not save Priority 1+2 latent observable parquet: {exc}")

print("Priority 2 observable columns:", len(priority2_cols))
print("Priority 1+2 observable columns:", len([c for c in observable_df.columns if c != "event_index"]))
print("Priority 1+2 merged rows:", len(latent_observable_df))

In [ ]:
### Priority 2 Validation, Correlations, and Summary

def validate_priority2_observables(df):
    """Basic finite-range checks for shower and texture observables."""
    checks = []
    for name in [c for c in df.columns if c.startswith("lund_n_") or c == "lund_tree_depth"]:
        vals = pd.to_numeric(df[name], errors="coerce")
        checks.append({"check": f"{name}_nonnegative", "passed": bool((vals.dropna() >= 0).all()), "n_checked": int(vals.notna().sum())})
    for name in ["lund_hardest_z", "lund_first_z", "subjet2_z1", "subjet2_z2", "subjet3_z1", "subjet3_z2", "subjet3_z3"]:
        if name in df.columns:
            vals = pd.to_numeric(df[name], errors="coerce").dropna()
            checks.append({"check": f"{name}_range", "passed": bool(((vals >= 0) & (vals <= 1)).all()), "n_checked": int(len(vals))})
    for name in [c for c in df.columns if c.startswith("fft_radial_bin_") or c in ["fft_low_freq_frac", "fft_high_freq_frac"]]:
        vals = pd.to_numeric(df[name], errors="coerce").dropna()
        checks.append({"check": f"{name}_unit_interval", "passed": bool(((vals >= -1e-9) & (vals <= 1 + 1e-9)).all()), "n_checked": int(len(vals))})
    for name in [c for c in df.columns if "_energy" in c and (c.startswith("wavelet_") or c.startswith("fft_"))]:
        vals = pd.to_numeric(df[name], errors="coerce").dropna()
        checks.append({"check": f"{name}_nonnegative", "passed": bool((vals >= -1e-12).all()), "n_checked": int(len(vals))})
    return pd.DataFrame(checks)


priority2_validation_df = validate_priority2_observables(priority2_observable_df)
priority2_validation_df.to_csv(OBS_TABLE_DIR / "priority2_validation_checks.csv", index=False)
display(priority2_validation_df.head(20))
if not bool(priority2_validation_df["passed"].all()):
    warnings.warn("Some Priority 2 validation checks failed. Inspect priority2_validation_checks.csv before interpreting correlations.")

observable_names = [name for name in observable_metadata_df["observable_name"] if name in latent_observable_df.columns]
primary_observable_names = observable_metadata_df.loc[
    (observable_metadata_df["observable_name"].isin(observable_names)) &
    (observable_metadata_df["family"] != "diagnostic") &
    (~observable_metadata_df["observable_name"].str.endswith(("cos_phase", "sin_phase"))),
    "observable_name",
].tolist()
diagnostic_observable_names = observable_metadata_df.loc[
    (observable_metadata_df["observable_name"].isin(observable_names)) &
    (observable_metadata_df["family"] == "diagnostic"),
    "observable_name",
].tolist()
priority2_primary_names = observable_metadata_df.loc[
    (observable_metadata_df["observable_name"].isin(observable_names)) &
    (observable_metadata_df["family"].isin(["shower", "texture"])),
    "observable_name",
].tolist()

priority12_raw_corr = correlation_matrix(latent_observable_df, latent_correlation_feature_names, primary_observable_names, method="pearson")
priority12_raw_corr.to_csv(OBS_TABLE_DIR / "latent_observable_corr_priority1_plus_priority2_raw.csv")
priority12_raw_family_heatmap_summary = save_family_latent_corr_heatmaps(
    priority12_raw_corr,
    primary_observable_names,
    observable_metadata_df,
    path_prefix="latent_observable_corr_priority1_plus_priority2_raw",
    title_prefix="Latent mu plus sigma_18 vs Priority 1+2 raw observables",
    top_n=40,
    residualized=False,
)

priority2_raw_corr = correlation_matrix(latent_observable_df, latent_correlation_feature_names, priority2_primary_names, method="pearson")
priority2_raw_corr.to_csv(OBS_TABLE_DIR / "latent_observable_corr_priority2_raw.csv")
if len(priority2_primary_names) > 0:
    priority2_family_heatmap_summary = save_family_latent_corr_heatmaps(
        priority2_raw_corr,
        priority2_primary_names,
        observable_metadata_df,
        path_prefix="latent_observable_corr_priority2_raw",
        title_prefix="Latent mu plus sigma_18 vs Priority 2 raw observables",
        top_n=40,
        residualized=False,
    )
else:
    priority2_family_heatmap_summary = pd.DataFrame()

priority12_residual_df = residualize_against_reference(latent_observable_df, primary_observable_names, reference_col=residual_reference_col)
priority12_residual_df.insert(0, "event_index", latent_observable_df["event_index"].to_numpy())
priority12_residual_df.to_csv(OBS_TABLE_DIR / f"observables_priority1_plus_priority2_residualized_{residual_reference_col}.csv", index=False)
priority12_latent_residual_df = pd.concat([
    latent_observable_df[["event_index"] + latent_correlation_feature_names].reset_index(drop=True),
    priority12_residual_df.drop(columns=["event_index"]).reset_index(drop=True),
], axis=1)
priority12_residual_names = [c for c in priority12_residual_df.columns if c != "event_index"]
priority12_resid_corr = correlation_matrix(priority12_latent_residual_df, latent_correlation_feature_names, priority12_residual_names, method="pearson")
priority12_resid_corr.to_csv(OBS_TABLE_DIR / f"latent_observable_corr_priority1_plus_priority2_residualized_{residual_reference_col}.csv")
priority12_resid_family_heatmap_summary = save_family_latent_corr_heatmaps(
    priority12_resid_corr,
    priority12_residual_names,
    observable_metadata_df,
    path_prefix=f"latent_observable_corr_priority1_plus_priority2_residualized_{residual_reference_col}",
    title_prefix=f"Latent mu plus sigma_18 vs Priority 1+2 observables residualized against {residual_reference_col}",
    top_n=40,
    residualized=True,
)

priority2_family_counts = observable_metadata_df[
    observable_metadata_df["observable_name"].isin(priority2_cols)
].groupby("family")["observable_name"].nunique().sort_values(ascending=False)
priority2_nan = priority2_observable_df[priority2_cols].isna().mean().sort_values(ascending=False)
priority2_raw_long = priority2_raw_corr.abs().stack().sort_values(ascending=False).head(30).reset_index()
priority2_raw_long.columns = ["latent", "observable", "abs_corr"]
priority12_resid_long = priority12_resid_corr.abs().stack().sort_values(ascending=False).head(30).reset_index()
priority12_resid_long.columns = ["latent", "observable", "abs_corr"]

priority2_summary_payload = {
    "priority2_family_counts": priority2_family_counts.to_dict(),
    "priority2_rows": int(len(priority2_observable_df)),
    "priority2_shower_rows_with_splits": int(pd.to_numeric(priority2_observable_df.get("lund_n_splits", pd.Series(dtype=float)), errors="coerce").notna().sum()),
    "priority2_config_hash": priority2_config_hash,
    "residualized_against": residual_reference_col,
    "cache_locations": {
        "priority2_observables_parquet": str(priority2_obs_cache_path),
        "priority2_observables_csv": str(priority2_obs_cache_csv_path),
        "priority2_lund_histograms_parquet": str(priority2_lund_hist_cache_path),
        "priority2_lund_histograms_csv": str(priority2_lund_hist_cache_csv_path),
    },
}
(OBS_TABLE_DIR / "priority2_summary.json").write_text(json.dumps(priority2_summary_payload, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(priority2_summary_payload, indent=2, ensure_ascii=False))
print("\nPriority 2 observable count by family:")
display(priority2_family_counts.to_frame("n_observables"))
print("\nPriority 2 largest NaN fractions:")
display(priority2_nan.head(20).to_frame("nan_fraction"))
print("\nStrongest Priority 2 raw latent-observable correlations:")
display(priority2_raw_long)
print(f"\nStrongest Priority 1+2 latent-observable correlations after residualizing against {residual_reference_col}:")
display(priority12_resid_long)
print("\nPriority 1+2 raw family heatmaps:")
display(priority12_raw_family_heatmap_summary[["family", "n_available", "n_plotted", "selected_observables"]])
print(f"\nPriority 1+2 residualized-against-{residual_reference_col} family heatmaps:")
display(priority12_resid_family_heatmap_summary[["family", "n_available", "n_plotted", "selected_observables"]])
print("\nSaved Priority 2 observables:", priority2_obs_cache_csv_path)
print("Saved Priority 1+2 raw correlations:", OBS_TABLE_DIR / "latent_observable_corr_priority1_plus_priority2_raw.csv")
print("Saved Priority 1+2 residualized correlations:", OBS_TABLE_DIR / f"latent_observable_corr_priority1_plus_priority2_residualized_{residual_reference_col}.csv")


## Priority 3 EFP, Soft Drop, and Regression Extension

This section adds the heavier Priority 3 analysis while keeping optional dependencies non-blocking.

Implemented here:

- optional connected Energy Flow Polynomials through the `energyflow` package, followed by EFP-block PCA;
- additional higher-point ECF extensions for extra beta values;
- Soft Drop-like recursive declustering observables using the existing C/A fallback tree, explicitly marked as a fallback rather than FastJet contrib SoftDrop;
- family-level observable PCs;
- Ridge regression diagnostics for `observable <- latent vector` and `latent dimension <- observable family PCs`, with optional GradientBoostingRegressor if enabled and sklearn is available.


In [ ]:
### Optional Priority 3 package installs
# Run this cell only if you want to enable optional EFP/regression packages in this notebook kernel.
# %pip install energyflow scikit-learn PyWavelets


In [ ]:
### Priority 3 Configuration

try:
    import energyflow as ef
    ENERGYFLOW_AVAILABLE = True
except Exception as exc:
    ef = None
    ENERGYFLOW_AVAILABLE = False
    warnings.warn(f"energyflow is not available; EFP raw features and EFP PCs will be skipped. Reason: {exc}")

try:
    from sklearn.linear_model import RidgeCV
    from sklearn.ensemble import GradientBoostingRegressor
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import r2_score
    SKLEARN_AVAILABLE = True
except Exception as exc:
    RidgeCV = None
    GradientBoostingRegressor = None
    train_test_split = None
    r2_score = None
    SKLEARN_AVAILABLE = False
    warnings.warn(f"sklearn is not available; regression diagnostics will be skipped. Reason: {exc}")

priority3_cache_version = "priority3_efp_softdrop_regression_v1"
if "residual_reference_col" not in globals():
    residual_reference_col = "girth"
if "residual_reference_suffix" not in globals():
    residual_reference_suffix = residual_suffix(residual_reference_col) if "residual_suffix" in globals() else "resid_girth"

efp_dmax = 5
efp_beta = 1.0
efp_pc_count = 10
efp_max_jets = min(int(expensive_max_jets), 5000)
max_efp_pixels = 40

p3_e3_beta_values = [0.25, 0.75, 1.5, 3.0, 4.0]
p3_max_e3_pixels = max_e3_pixels

softdrop_settings = [
    {"zcut": 0.1, "beta": 0.0},
    {"zcut": 0.1, "beta": 1.0},
    {"zcut": 0.05, "beta": 2.0},
]

family_pc_count = 10
regression_random_seed = random_seed
regression_test_size = 0.25
regression_top_observables = 80
run_gradient_boosting_regression = False

priority3_config = {
    "base_observable_config_hash": observable_config_hash,
    "priority2_config_hash": priority2_config_hash,
    "priority3_cache_version": priority3_cache_version,
    "efp_dmax": efp_dmax,
    "efp_beta": efp_beta,
    "efp_pc_count": efp_pc_count,
    "efp_max_jets": efp_max_jets,
    "max_efp_pixels": max_efp_pixels,
    "p3_e3_beta_values": p3_e3_beta_values,
    "p3_max_e3_pixels": p3_max_e3_pixels,
    "softdrop_settings": softdrop_settings,
    "family_pc_count": family_pc_count,
    "regression_random_seed": regression_random_seed,
    "regression_test_size": regression_test_size,
    "regression_top_observables": regression_top_observables,
    "run_gradient_boosting_regression": run_gradient_boosting_regression,
    "residualized_against": residual_reference_col,
}
priority3_config_hash = hashlib.sha256(json.dumps(priority3_config, sort_keys=True).encode("utf-8")).hexdigest()[:16]
print("energyflow available:", ENERGYFLOW_AVAILABLE)
print("sklearn available:", SKLEARN_AVAILABLE)
print("Priority 3 config hash:", priority3_config_hash)
print("Priority 3 EFP max jets:", efp_max_jets)
print("Priority 3 residualized against:", residual_reference_col)


In [ ]:
### Priority 3 Observable and Regression Functions

def compute_e2_beta_direct(z, x, y, beta):
    """Compute e2_beta for already selected active pixels."""
    d, w = pairwise_distances_and_weights(z, x, y)
    return float(np.sum(w * (d ** beta))) if len(d) else 0.0


def compute_priority3_ecf_observables(z_img, x_grid, y_grid):
    """Compute additional approximate e3, C2, and D2 values for Priority 3 beta extensions."""
    z, x, y = select_active_pixels(z_img, x_grid, y_grid, max_pixels=p3_max_e3_pixels)
    out = {
        "p3_e3_selected_energy_fraction": float(np.sum(z)),
        "p3_e3_n_selected_pixels": int(len(z)),
    }
    for beta in p3_e3_beta_values:
        label = beta_label(beta)
        e2_value = compute_e2_beta_direct(z, x, y, beta)
        e3_value = compute_e3_beta(z, x, y, beta) if len(z) >= 3 else 0.0
        out[f"p3_e3_beta_{label}"] = e3_value
        out[f"p3_C2_beta_{label}"] = np.nan if e2_value <= ecf_denominator_eps else float(e3_value / (e2_value ** 2))
        out[f"p3_D2_beta_{label}"] = np.nan if e2_value <= ecf_denominator_eps else float(e3_value / (e2_value ** 3))
    return out


def softdrop_label(setting):
    """Stable suffix for one Soft Drop setting."""
    return f"zcut{beta_label(setting['zcut'])}_beta{beta_label(setting['beta'])}"


def compute_softdrop_fallback_observables(z_img, x_grid, y_grid):
    """Compute Soft Drop-like observables using the fallback C/A tree.

    This is not FastJet contrib SoftDrop. It recursively declusters the fallback C/A tree along the hardest branch,
    applies z > zcut * Delta^beta with Delta already normalized by AK4 R, and estimates rho_g as z_g(1-z_g)R_g^2.
    """
    nodes, root_id, z, x, y = build_ca_tree_fallback(z_img, x_grid, y_grid)
    out = {}
    for setting in softdrop_settings:
        label = softdrop_label(setting)
        for suffix in ["zg", "rg_over_R", "rho_g", "removed_energy_fraction", "n_sd"]:
            out[f"softdrop_{label}_{suffix}"] = np.nan
        out[f"softdrop_{label}_n_sd"] = 0
        if root_id is None:
            continue
        current = root_id
        original_pt = max(float(nodes[root_id]["pt"]), np.finfo(np.float64).tiny)
        removed = 0.0
        accepted = []
        first_pass = None
        while current is not None:
            node = nodes[current]
            left = node["left"]
            right = node["right"]
            if left is None or right is None:
                break
            a = nodes[left]
            b = nodes[right]
            denom = float(a["pt"] + b["pt"])
            if denom <= 0:
                break
            delta = float(math.hypot(a["x"] - b["x"], a["y"] - b["y"]))
            zg = float(min(a["pt"], b["pt"]) / denom)
            threshold = float(setting["zcut"] * (max(delta, np.finfo(np.float64).tiny) ** setting["beta"]))
            hard_id, soft_id = (left, right) if a["pt"] >= b["pt"] else (right, left)
            if zg > threshold:
                accepted.append((zg, delta))
                if first_pass is None:
                    first_pass = (zg, delta, removed / original_pt)
                    break
            else:
                removed += float(nodes[soft_id]["pt"])
                current = hard_id
        out[f"softdrop_{label}_n_sd"] = int(len(accepted))
        if first_pass is not None:
            zg, rg, removed_frac = first_pass
            out[f"softdrop_{label}_zg"] = float(zg)
            out[f"softdrop_{label}_rg_over_R"] = float(rg)
            out[f"softdrop_{label}_rho_g"] = float(zg * (1.0 - zg) * rg ** 2)
            out[f"softdrop_{label}_removed_energy_fraction"] = float(removed_frac)
        else:
            out[f"softdrop_{label}_removed_energy_fraction"] = float(removed / original_pt) if original_pt > 0 else np.nan
    return out


def initialize_efp_set():
    """Initialize connected EFPs through energyflow when available."""
    if not ENERGYFLOW_AVAILABLE:
        return None, [], pd.DataFrame()
    try:
        efp_set = ef.EFPSet(("d<=", efp_dmax), ("c==", 1), measure="hadr", beta=efp_beta)
        graphs = list(getattr(efp_set, "graphs", lambda: [])())
        try:
            n_efps = len(efp_set)
        except Exception:
            n_efps = len(getattr(efp_set, "efps", [])) or len(graphs)
        rows = []
        names = []
        for i in range(int(n_efps)):
            graph = graphs[i] if i < len(graphs) else None
            graph_id = str(graph) if graph is not None else f"index_{i:03d}"
            degree = np.nan
            n_vertices = np.nan
            n_edges = np.nan
            if graph is not None:
                try:
                    edges = list(graph)
                    n_edges = len(edges)
                    vertices = sorted(set([v for edge in edges for v in edge])) if edges else []
                    n_vertices = len(vertices)
                    degree = n_edges
                except Exception:
                    pass
            name = f"efp_{i:03d}"
            names.append(name)
            rows.append({
                "observable_name": name,
                "degree": degree,
                "n_vertices": n_vertices,
                "n_edges": n_edges,
                "graph_identifier": graph_id,
                "connected": True,
                "beta": efp_beta,
                "dmax": efp_dmax,
            })
        return efp_set, names, pd.DataFrame(rows)
    except Exception as exc:
        warnings.warn(f"Could not initialize energyflow EFPSet; EFPs will be skipped. Reason: {exc}")
        return None, [], pd.DataFrame()


def compute_efp_row(z_img, x_grid, y_grid, efp_set, efp_names):
    """Compute one row of raw EFP values through energyflow, or an empty row if unavailable."""
    if efp_set is None or not efp_names:
        return {}
    z, x, y = select_active_pixels(z_img, x_grid, y_grid, threshold=lund_pixel_threshold, max_pixels=max_efp_pixels)
    if len(z) == 0:
        return {name: np.nan for name in efp_names}
    event = np.column_stack([z, x, y])
    try:
        values = np.asarray(efp_set.compute(event), dtype=np.float64).ravel()
    except Exception as exc:
        warnings.warn(f"EFP computation failed for one event; filling NaN. Reason: {exc}")
        return {name: np.nan for name in efp_names}
    out = {}
    for i, name in enumerate(efp_names):
        out[name] = float(values[i]) if i < len(values) and np.isfinite(values[i]) else np.nan
    return out


def build_family_pc_dataframe(input_df, metadata_df, n_components=family_pc_count):
    """Build family-level PCA scores for non-diagnostic observable blocks."""
    pc_df = pd.DataFrame({"event_index": input_df["event_index"].to_numpy()})
    meta_parts = []
    families = sorted(metadata_df.loc[~metadata_df["family"].isin(["diagnostic", "efp_raw"]), "family"].dropna().unique())
    for family in families:
        cols = metadata_df.loc[
            (metadata_df["family"] == family) &
            (metadata_df["observable_name"].isin(input_df.columns)),
            "observable_name",
        ].tolist()
        cols = [c for c in cols if pd.to_numeric(input_df[c], errors="coerce").nunique(dropna=True) > 1]
        if len(cols) < 2:
            continue
        block_df = input_df[["event_index"] + cols].copy()
        family_prefix = "family_" + re.sub(r"[^0-9A-Za-z]+", "_", family).strip("_")
        pcs, meta = compute_pca_block(block_df, cols, n_components=n_components, prefix=family_prefix)
        pc_df = pc_df.merge(pcs, on="event_index", how="left")
        if not meta.empty:
            meta.insert(0, "family", family)
            meta_parts.append(meta)
    meta_df = pd.concat(meta_parts, ignore_index=True) if meta_parts else pd.DataFrame()
    return pc_df, meta_df


def finite_regression_frame(df, x_cols, y_col):
    """Return finite X, y arrays and the corresponding row index for regression."""
    cols = [c for c in x_cols if c in df.columns]
    if y_col not in df.columns or not cols:
        return None, None, None, []
    X = df[cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
    y = pd.to_numeric(df[y_col], errors="coerce").to_numpy(dtype=np.float64)
    mask = np.isfinite(y) & np.isfinite(X).all(axis=1)
    if mask.sum() < 20:
        return None, None, None, cols
    return X[mask], y[mask], np.flatnonzero(mask), cols


def evaluate_ridge_regressions(df, x_cols, y_cols, *, task_name):
    """Evaluate RidgeCV train/test R2 for multiple scalar targets."""
    rows = []
    if not SKLEARN_AVAILABLE:
        return pd.DataFrame(rows)
    for y_col in y_cols:
        X, y, _, used_x_cols = finite_regression_frame(df, x_cols, y_col)
        if X is None:
            rows.append({"task": task_name, "target": y_col, "model": "ridge", "r2": np.nan, "n_samples": 0, "n_features": len(used_x_cols)})
            continue
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=regression_test_size, random_state=regression_random_seed)
        x_mean = X_train.mean(axis=0)
        x_std = X_train.std(axis=0)
        keep = np.isfinite(x_std) & (x_std > 1e-12)
        if keep.sum() == 0:
            rows.append({"task": task_name, "target": y_col, "model": "ridge", "r2": np.nan, "n_samples": int(len(y)), "n_features": 0})
            continue
        X_train_z = (X_train[:, keep] - x_mean[keep]) / x_std[keep]
        X_test_z = (X_test[:, keep] - x_mean[keep]) / x_std[keep]
        y_mean = float(np.mean(y_train))
        y_std = float(np.std(y_train))
        if y_std <= 1e-12:
            rows.append({"task": task_name, "target": y_col, "model": "ridge", "r2": np.nan, "n_samples": int(len(y)), "n_features": int(keep.sum())})
            continue
        model = RidgeCV(alphas=np.logspace(-4, 4, 17))
        model.fit(X_train_z, (y_train - y_mean) / y_std)
        pred = model.predict(X_test_z) * y_std + y_mean
        rows.append({"task": task_name, "target": y_col, "model": "ridge", "r2": float(r2_score(y_test, pred)), "n_samples": int(len(y)), "n_features": int(keep.sum()), "alpha": float(model.alpha_)})
        if run_gradient_boosting_regression:
            gb = GradientBoostingRegressor(random_state=regression_random_seed)
            gb.fit(X_train_z, y_train)
            gb_pred = gb.predict(X_test_z)
            rows.append({"task": task_name, "target": y_col, "model": "gradient_boosting", "r2": float(r2_score(y_test, gb_pred)), "n_samples": int(len(y)), "n_features": int(keep.sum()), "alpha": np.nan})
    return pd.DataFrame(rows)


def run_priority3_sanity_tests():
    """Run lightweight checks for Priority 3 fallback observables."""
    center = synthetic_image([(obs_img_size // 2, obs_img_size // 2, 1.0)])
    two = synthetic_image([(obs_img_size // 2, obs_img_size // 2 - 4, 0.5), (obs_img_size // 2, obs_img_size // 2 + 4, 0.5)])
    center_prob, _ = prepare_probability_image(center)
    two_prob, _ = prepare_probability_image(two)
    center_ecf = compute_priority3_ecf_observables(center_prob, X_GRID, Y_GRID)
    two_sd = compute_softdrop_fallback_observables(two_prob, X_GRID, Y_GRID)
    assert center_ecf["p3_e3_n_selected_pixels"] == 1
    assert all(v == 0.0 for k, v in center_ecf.items() if k.startswith("p3_e3_beta_"))
    zg_cols = [k for k in two_sd if k.endswith("_zg")]
    assert any(np.isclose(two_sd[k], 0.5, equal_nan=False) for k in zg_cols)
    return pd.DataFrame([{**center_ecf, **two_sd}])


priority3_sanity_df = run_priority3_sanity_tests()
display(priority3_sanity_df.head())


In [ ]:
### Compute and Cache Priority 3 Observables

priority3_metadata_rows = []
for beta in p3_e3_beta_values:
    label = beta_label(beta)
    for prefix in ["p3_e3_beta", "p3_C2_beta", "p3_D2_beta"]:
        add_metadata(priority3_metadata_rows, f"{prefix}_{label}", "energy_flow", "p3_e3_c2_d2", f"Priority 3 approximate {prefix} extension", expensive=True, parameters={"beta": beta, "max_pixels": p3_max_e3_pixels})
for name in ["p3_e3_selected_energy_fraction", "p3_e3_n_selected_pixels"]:
    add_metadata(priority3_metadata_rows, name, "diagnostic", "priority3_approximation", f"Priority 3 ECF approximation diagnostic {name}", irc_safe=False, pixelization_sensitive=True, expensive=True)
for setting in softdrop_settings:
    label = softdrop_label(setting)
    for suffix in ["zg", "rg_over_R", "rho_g", "removed_energy_fraction", "n_sd"]:
        add_metadata(priority3_metadata_rows, f"softdrop_{label}_{suffix}", "shower", "softdrop_fallback", "Soft Drop-like fallback observable from C/A pixel tree", expensive=True, parameters=setting)
for i in range(1, efp_pc_count + 1):
    add_metadata(priority3_metadata_rows, f"efp_pc{i:02d}", "efp", "efp_pca", "PCA component of connected Energy Flow Polynomial block", expensive=True, parameters={"dmax": efp_dmax, "beta": efp_beta})

efp_set, efp_names, efp_graph_metadata_df = initialize_efp_set()
if not efp_graph_metadata_df.empty:
    efp_graph_metadata_df.to_csv(OBS_TABLE_DIR / "priority3_efp_graph_metadata.csv", index=False)
for name in efp_names:
    add_metadata(priority3_metadata_rows, name, "efp_raw", "energyflow_package", "Raw connected Energy Flow Polynomial from energyflow package", expensive=True, parameters={"dmax": efp_dmax, "beta": efp_beta})

priority3_metadata_df = pd.DataFrame(priority3_metadata_rows).drop_duplicates("observable_name")
observable_metadata_df = pd.concat([observable_metadata_df, priority3_metadata_df], ignore_index=True).drop_duplicates("observable_name", keep="last")
observable_metadata_df.to_csv(OBS_TABLE_DIR / "observable_metadata.csv", index=False)

priority3_obs_cache_path = OBS_CACHE_DIR / f"priority3_observables_{priority3_config_hash}.parquet"
priority3_obs_cache_csv_path = OBS_CACHE_DIR / f"priority3_observables_{priority3_config_hash}.csv"
priority3_efp_raw_cache_path = OBS_CACHE_DIR / f"priority3_efp_raw_{priority3_config_hash}.parquet"
priority3_efp_raw_cache_csv_path = OBS_CACHE_DIR / f"priority3_efp_raw_{priority3_config_hash}.csv"

if priority3_obs_cache_path.exists():
    priority3_observable_df = pd.read_parquet(priority3_obs_cache_path)
    print("loaded Priority 3 observables from parquet cache:", priority3_obs_cache_path)
    if priority3_efp_raw_cache_path.exists():
        efp_raw_df = pd.read_parquet(priority3_efp_raw_cache_path)
    elif priority3_efp_raw_cache_csv_path.exists():
        efp_raw_df = pd.read_csv(priority3_efp_raw_cache_csv_path)
    else:
        efp_raw_df = pd.DataFrame({"event_index": priority3_observable_df["event_index"]})
elif priority3_obs_cache_csv_path.exists():
    priority3_observable_df = pd.read_csv(priority3_obs_cache_csv_path)
    print("loaded Priority 3 observables from CSV cache:", priority3_obs_cache_csv_path)
    if priority3_efp_raw_cache_path.exists():
        efp_raw_df = pd.read_parquet(priority3_efp_raw_cache_path)
    elif priority3_efp_raw_cache_csv_path.exists():
        efp_raw_df = pd.read_csv(priority3_efp_raw_cache_csv_path)
    else:
        efp_raw_df = pd.DataFrame({"event_index": priority3_observable_df["event_index"]})
else:
    if len(observable_event_indices) > efp_max_jets:
        efp_indices = set(rng.choice(np.arange(len(observable_event_indices)), size=int(efp_max_jets), replace=False).tolist())
        print(f"Priority 3 EFP modules run on {efp_max_jets} reproducible jets; other rows receive NaN for EFP columns")
    else:
        efp_indices = set(range(len(observable_event_indices)))
    rows = []
    efp_rows = []
    for local_idx, event_index in tqdm(list(enumerate(observable_event_indices)), desc="Compute Priority 3 observables"):
        z_img, prep = prepare_probability_image(normalized_images[local_idx])
        row = {"event_index": int(event_index)}
        efp_row = {"event_index": int(event_index)}
        if prep["valid"]:
            row.update(compute_priority3_ecf_observables(z_img, X_GRID, Y_GRID))
            row.update(compute_softdrop_fallback_observables(z_img, X_GRID, Y_GRID))
            if local_idx in efp_indices and efp_set is not None:
                efp_row.update(compute_efp_row(z_img, X_GRID, Y_GRID, efp_set, efp_names))
        rows.append(row)
        efp_rows.append(efp_row)
    priority3_raw_observable_df = pd.DataFrame(rows)
    efp_raw_df = pd.DataFrame(efp_rows)
    efp_feature_cols = [c for c in efp_names if c in efp_raw_df.columns]
    efp_pc_df, efp_pca_metadata_df = compute_pca_block(efp_raw_df, efp_feature_cols, n_components=efp_pc_count, prefix="efp")
    efp_pca_metadata_df.to_csv(OBS_TABLE_DIR / "priority3_efp_pca_metadata.csv", index=False)
    priority3_observable_df = priority3_raw_observable_df.merge(efp_pc_df, on="event_index", how="left")
    try:
        priority3_observable_df.to_parquet(priority3_obs_cache_path, index=False)
        efp_raw_df.to_parquet(priority3_efp_raw_cache_path, index=False)
        print("saved Priority 3 parquet caches")
    except Exception as exc:
        warnings.warn(f"Could not save Priority 3 parquet cache, saving CSV instead: {exc}")
    priority3_observable_df.to_csv(priority3_obs_cache_csv_path, index=False)
    efp_raw_df.to_csv(priority3_efp_raw_cache_csv_path, index=False)
    print("saved Priority 3 CSV caches:", priority3_obs_cache_csv_path, priority3_efp_raw_cache_csv_path)

priority3_cols = [c for c in priority3_observable_df.columns if c != "event_index"]
observable_df = observable_df.drop(columns=[c for c in priority3_cols if c in observable_df.columns], errors="ignore")
observable_df = observable_df.merge(priority3_observable_df, on="event_index", how="left")
latent_observable_df = latent_mu_df.merge(observable_df, on="event_index", how="inner")
latent_observable_df.to_csv(OBS_TABLE_DIR / "latent_observables_priority1_plus_priority2_plus_priority3.csv", index=False)
try:
    latent_observable_df.to_parquet(OBS_TABLE_DIR / "latent_observables_priority1_plus_priority2_plus_priority3.parquet", index=False)
except Exception as exc:
    warnings.warn(f"Could not save Priority 1+2+3 latent observable parquet: {exc}")

print("Priority 3 observable columns:", len(priority3_cols))
print("Priority 1+2+3 observable columns:", len([c for c in observable_df.columns if c != "event_index"]))
print("Priority 1+2+3 merged rows:", len(latent_observable_df))


In [ ]:
### Priority 3 Correlations and Regression Diagnostics

observable_names = [name for name in observable_metadata_df["observable_name"] if name in latent_observable_df.columns]
primary_observable_names = observable_metadata_df.loc[
    (observable_metadata_df["observable_name"].isin(observable_names)) &
    (~observable_metadata_df["family"].isin(["diagnostic", "efp_raw"])) &
    (~observable_metadata_df["observable_name"].str.endswith(("cos_phase", "sin_phase"))),
    "observable_name",
].tolist()
priority3_primary_names = observable_metadata_df.loc[
    (observable_metadata_df["observable_name"].isin(priority3_cols)) &
    (~observable_metadata_df["family"].isin(["diagnostic", "efp_raw"])),
    "observable_name",
].tolist()

priority123_raw_corr = correlation_matrix(latent_observable_df, latent_correlation_feature_names, primary_observable_names, method="pearson")
priority123_raw_corr.to_csv(OBS_TABLE_DIR / "latent_observable_corr_priority1_plus_priority2_plus_priority3_raw.csv")
priority123_raw_family_heatmap_summary = save_family_latent_corr_heatmaps(
    priority123_raw_corr,
    primary_observable_names,
    observable_metadata_df,
    path_prefix="latent_observable_corr_priority1_plus_priority2_plus_priority3_raw",
    title_prefix="Latent mu plus sigma_18 vs Priority 1+2+3 raw observables",
    top_n=40,
    residualized=False,
)

priority3_raw_corr = correlation_matrix(latent_observable_df, latent_correlation_feature_names, priority3_primary_names, method="pearson")
priority3_raw_corr.to_csv(OBS_TABLE_DIR / "latent_observable_corr_priority3_raw.csv")
priority3_raw_family_heatmap_summary = save_family_latent_corr_heatmaps(
    priority3_raw_corr,
    priority3_primary_names,
    observable_metadata_df,
    path_prefix="latent_observable_corr_priority3_raw",
    title_prefix="Latent mu plus sigma_18 vs Priority 3 raw observables",
    top_n=40,
    residualized=False,
)

priority123_residual_df = residualize_against_reference(latent_observable_df, primary_observable_names, reference_col=residual_reference_col)
priority123_residual_df.insert(0, "event_index", latent_observable_df["event_index"].to_numpy())
priority123_residual_df.to_csv(OBS_TABLE_DIR / f"observables_priority1_plus_priority2_plus_priority3_residualized_{residual_reference_col}.csv", index=False)
priority123_latent_residual_df = pd.concat([
    latent_observable_df[["event_index"] + latent_correlation_feature_names].reset_index(drop=True),
    priority123_residual_df.drop(columns=["event_index"]).reset_index(drop=True),
], axis=1)
priority123_residual_names = [c for c in priority123_residual_df.columns if c != "event_index"]
priority123_resid_corr = correlation_matrix(priority123_latent_residual_df, latent_correlation_feature_names, priority123_residual_names, method="pearson")
priority123_resid_corr.to_csv(OBS_TABLE_DIR / f"latent_observable_corr_priority1_plus_priority2_plus_priority3_residualized_{residual_reference_col}.csv")
priority123_resid_family_heatmap_summary = save_family_latent_corr_heatmaps(
    priority123_resid_corr,
    priority123_residual_names,
    observable_metadata_df,
    path_prefix=f"latent_observable_corr_priority1_plus_priority2_plus_priority3_residualized_{residual_reference_col}",
    title_prefix=f"Latent mu plus sigma_18 vs Priority 1+2+3 observables residualized against {residual_reference_col}",
    top_n=40,
    residualized=True,
)

family_pc_df, family_pc_metadata_df = build_family_pc_dataframe(latent_observable_df, observable_metadata_df, n_components=family_pc_count)
family_pc_df.to_csv(OBS_TABLE_DIR / "priority3_observable_family_pcs.csv", index=False)
family_pc_metadata_df.to_csv(OBS_TABLE_DIR / "priority3_observable_family_pc_metadata.csv", index=False)
family_pc_cols = [c for c in family_pc_df.columns if c != "event_index"]
regression_df = latent_observable_df.merge(family_pc_df, on="event_index", how="left")

if len(primary_observable_names):
    regression_scores = priority123_raw_corr[primary_observable_names].abs().replace([np.inf, -np.inf], np.nan).max(axis=0).dropna()
    selected_regression_observables = regression_scores.sort_values(ascending=False).head(regression_top_observables).index.tolist()
else:
    selected_regression_observables = []

latent_predicts_observable_r2 = evaluate_ridge_regressions(
    regression_df,
    latent_correlation_feature_names,
    selected_regression_observables,
    task_name="observable_from_latent",
)
latent_from_family_pc_r2 = evaluate_ridge_regressions(
    regression_df,
    family_pc_cols,
    latent_correlation_feature_names,
    task_name="latent_from_observable_family_pcs",
)
latent_predicts_observable_r2.to_csv(OBS_TABLE_DIR / "priority3_ridge_observable_from_latent_r2.csv", index=False)
latent_from_family_pc_r2.to_csv(OBS_TABLE_DIR / "priority3_ridge_latent_from_observable_family_pcs_r2.csv", index=False)

priority3_family_counts = observable_metadata_df[
    observable_metadata_df["observable_name"].isin(priority3_cols)
].groupby("family")["observable_name"].nunique().sort_values(ascending=False)
priority3_nan = priority3_observable_df[priority3_cols].isna().mean().sort_values(ascending=False)
priority123_raw_long = priority123_raw_corr.abs().stack().sort_values(ascending=False).head(30).reset_index()
priority123_raw_long.columns = ["latent", "observable", "abs_corr"]
priority123_resid_long = priority123_resid_corr.abs().stack().sort_values(ascending=False).head(30).reset_index()
priority123_resid_long.columns = ["latent", "observable", "abs_corr"]

priority3_summary_payload = {
    "priority3_family_counts": priority3_family_counts.to_dict(),
    "priority3_rows": int(len(priority3_observable_df)),
    "energyflow_available": bool(ENERGYFLOW_AVAILABLE),
    "sklearn_available": bool(SKLEARN_AVAILABLE),
    "efp_raw_columns": int(len([c for c in efp_raw_df.columns if c != "event_index"])),
    "family_pc_columns": int(len(family_pc_cols)),
    "regression_observable_targets": int(len(selected_regression_observables)),
    "priority3_config_hash": priority3_config_hash,
    "residualized_against": residual_reference_col,
    "cache_locations": {
        "priority3_observables_parquet": str(priority3_obs_cache_path),
        "priority3_observables_csv": str(priority3_obs_cache_csv_path),
        "priority3_efp_raw_parquet": str(priority3_efp_raw_cache_path),
        "priority3_efp_raw_csv": str(priority3_efp_raw_cache_csv_path),
    },
}
(OBS_TABLE_DIR / "priority3_summary.json").write_text(json.dumps(priority3_summary_payload, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(priority3_summary_payload, indent=2, ensure_ascii=False))
print("\nPriority 3 observable count by family:")
display(priority3_family_counts.to_frame("n_observables"))
print("\nPriority 3 largest NaN fractions:")
display(priority3_nan.head(20).to_frame("nan_fraction"))
print("\nStrongest Priority 1+2+3 raw latent-observable correlations:")
display(priority123_raw_long)
print(f"\nStrongest Priority 1+2+3 latent-observable correlations after residualizing against {residual_reference_col}:")
display(priority123_resid_long)
print("\nTop Ridge R2 for observable <- latent:")
display(latent_predicts_observable_r2.sort_values("r2", ascending=False).head(20) if not latent_predicts_observable_r2.empty else latent_predicts_observable_r2)
print("\nTop Ridge R2 for latent <- observable family PCs:")
display(latent_from_family_pc_r2.sort_values("r2", ascending=False).head(20) if not latent_from_family_pc_r2.empty else latent_from_family_pc_r2)
print("\nSaved Priority 3 observables:", priority3_obs_cache_csv_path)
print("Saved Priority 1+2+3 raw correlations:", OBS_TABLE_DIR / "latent_observable_corr_priority1_plus_priority2_plus_priority3_raw.csv")
print("Saved Priority 1+2+3 residualized correlations:", OBS_TABLE_DIR / f"latent_observable_corr_priority1_plus_priority2_plus_priority3_residualized_{residual_reference_col}.csv")
